# 02 · Data quality

**Goal:** review the quality of the **training** set generated in notebook 01 (nulls, categories, formats and outliers) and fix whatever is needed so that the following notebooks work with reliable data.

**How we work:** each block follows three steps:
1. **Observation:** tables and charts to understand how each problem behaves (for example, whether nulls or outliers are related to default).
2. **Decision:** what is done and why, based on what was observed.
3. **Fix:** the code that applies the decision.

> **General rule:** everything learned from the data (limits, imputation values...) is computed **only on training**. It is then applied in the same way to validation and to the data received by the API, so that no information leaks from the validation set.

> **Note on names:** column names, category values and code identifiers are kept in Spanish, as in the data files and the API used across the project. The dictionary in section 3.1 explains every column.

<a id="s0"></a>

## Decision log

*(Filled in as each point is decided.)*

| # | Variable | Decision | Main reason |
|---|---|---|---|
| D1 | `num_meses_desde_ult_retraso` | Convert into bands (`sin_retraso`, `0-24`, `25-48`, `>48`) | The null means "no recorded late payment" and no number represents that |
| D2 | `antigüedad_empleo` | Null → own category `sin_empleo_declarado` | The null identifies a distinct profile with the highest default rate |
| D3 | `empleo` | Replace with `sector_empleo` (keyword rules) | Free text is unusable, but the sector carries signal beyond the grade |
| D4 | `porc_uso_revolving` | Null → training median (55.2 %) | Without a revolving limit utilisation cannot be computed; the median does not invent a low or extreme risk and is applied the same way in the API |
| D5 | `porc_uso_tarjetas`, `porc_tarjetas_75p` | Logical rules + training median by decile of `porc_uso_revolving` | Highly correlated with revolving; halves the error without impossible values and is a 10-row table in the API |
| D6 | `num_hipotecas` | Null → training median by `vivienda` (fallback: global median) | Null due to the date (Jan–Apr 2012), not the borrower; housing triples the exact hits versus the global median |

## Table of contents

- [Decision log](#s0)
- [1 · Import libraries](#s1)
- [2 · Load the data](#s2)
- [3 · Overview](#s3)
  - [3.1 · Data dictionary](#s3-1)
- [4 · Nulls](#s4)
  - [4.1 · Which variables have nulls and how many?](#s4-1)
  - [4.2 · Is the null related to default?](#s4-2)
  - [4.3 · Do nulls depend on the issue year?](#s4-3)
  - [4.4 · Variable by variable](#s4-4)
    - [4.4.1 · `num_meses_desde_ult_retraso` (D1)](#s4-4-1)
    - [4.4.2 · `antigüedad_empleo` (D2)](#s4-4-2)
    - [4.4.3 · `empleo` (D3)](#s4-4-3)
    - [4.4.4 · `porc_uso_revolving` (D4)](#s4-4-4)
    - [4.4.5 · `porc_uso_tarjetas` and `porc_tarjetas_75p` (D5)](#s4-4-5)
    - [4.4.6 · `num_hipotecas` (D6)](#s4-4-6)
  - [4.5 · Final null check](#s4-5)
  - [4.6 · Summary of the nulls block](#s4-6)
- *5 · Categories and text (pending)*
- *6 · Outliers (pending)*
- *7 · Save the corrected dataset (pending)*

> Each section starts with a **blue band**. Below every result there is an explanation of **what is done** and **what we see**.

<a id="s1"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">1 · IMPORT LIBRARIES</span>

</div>

In [ ]:
# --- Libraries -------------------------------------------------------------
from pathlib import Path          # OS-independent file paths
import numpy as np                # numerical computing
import pandas as pd               # tables (DataFrames)
import matplotlib.pyplot as plt   # charts
import seaborn as sns             # statistical charts on top of matplotlib

# --- Display options -------------------------------------------------------
pd.set_option('display.float_format', lambda x: '%.3f' % x)  # no scientific notation
pd.set_option('display.max_columns', 60)                     # show every column
np.set_printoptions(suppress=True)
sns.set_theme(style='whitegrid')                             # chart style

print("✅ Libraries imported successfully")

**What is done:** the libraries are imported and the display of tables and charts is configured.

**What we see:** the message confirms that every library is installed in the environment.

<a id="s2"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">2 · LOAD THE DATA</span>

</div>

In [ ]:
# Path of the training set (relative to this notebook's folder: 03_notebooks/en)
RUTA_ENTRENAMIENTO = Path('../../02_datos/03_Entrenamiento/train.pkl')

# Load the training set generated in notebook 01
df = pd.read_pickle(RUTA_ENTRENAMIENTO)
print(f"Loans: {df.shape[0]:,} · Columns: {df.shape[1]}")

**What is done:** the **training** set generated by notebook 01 is loaded.

**What we see:** **443,469 loans and 34 columns**. The validation set is not looked at in this notebook: everything decided here is learned from training only.

In [ ]:
# Default flag (True = 'Charged Off'). It is used throughout the notebook to check whether
# a quality problem (nulls, outliers...) is related to risk.
impago = df['estado'].eq('Charged Off')

# Overall default rate: the reference against which each group will be compared
TASA_IMPAGO_GLOBAL = impago.mean()
print(f"Overall default rate: {TASA_IMPAGO_GLOBAL:.1%}")

**What is done:** the default flag is created (`True` if the loan ended *Charged Off*) and the overall default rate is computed.

**What we see:** **14.8 %** of the loans ended in default. It is the reference each group is compared against throughout the notebook.

<a id="s3"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">3 · OVERVIEW</span>

</div>

In [ ]:
# Data type, number of non-null values and memory of each column
df.info()

**What is done:** for each column, its data type and number of non-null values are shown.

**What we see:** there are 22 numeric columns (`float64`/`int64`), 9 text columns (`str`) and 2 dates. Columns with fewer than 443,469 non-null values have nulls: `empleo`, `antigüedad_empleo`, `num_hipotecas`, `porc_uso_revolving`, `porc_uso_tarjetas`, `porc_tarjetas_75p` and `num_meses_desde_ult_retraso` (section 4).

In [ ]:
# First rows shown vertically (one row per column) to see the format of each variable
df.head(3).T

**What is done:** 3 loans are shown vertically (one row per column) to see the actual format of each variable.

**What we see:** some numeric variables are stored as text, such as `num_cuotas` (`"36 months"`) or `antigüedad_empleo` (`"9 years"`); they will be handled in the categories block. `NaN` means a null value.

In [ ]:
# Number of distinct values per column (from fewest to most)
df.nunique().sort_values().to_frame('valores_distintos')

**What is done:** we count how many distinct values (`valores_distintos`) each column has.

**What we see:** `num_cuotas` and `estado` only have 2 values (36/60 months; *Fully Paid*/*Charged Off*). Categorical variables have few values (`vivienda` 6, `finalidad` 14), except `empleo`, with **170,605** distinct values because it is free text. `id_cliente` has one value per loan.

In [ ]:
# Duplicates: does any loan appear more than once?
print(f"Duplicated identifiers: {df['id_cliente'].duplicated().sum()}")

**What is done:** we check whether any loan identifier appears more than once.

**What we see:** there are no duplicates: each row is a different loan.

<a id="s3-1"></a>

### 3.1 · Data dictionary

Meaning of the 34 columns of the dataset (more detail in notebook 01, section 7.2). The original Lending Club name is in brackets.

| Column | Meaning |
|---|---|
| `id_cliente` (`id`) | Unique identifier of the loan. |
| `empleo` (`emp_title`) | Job title stated by the borrower (free text). |
| `antigüedad_empleo` (`emp_length`) | Years in the current job, from `< 1 year` to `10+ years`. |
| `ingresos` (`annual_inc`) | Stated annual income ($). |
| `ingresos_verificados` (`verification_status`) | Whether Lending Club checked the income. |
| `vivienda` (`home_ownership`) | `RENT`, `MORTGAGE` (owned with a mortgage), `OWN` (owned outright) or others. |
| `dti` (`dti`) | Monthly debt payments (excluding the mortgage and this loan) / monthly income, in %. |
| `finalidad` (`purpose`) | What the loan is for. |
| `principal` (`funded_amnt`) | Amount lent ($). |
| `num_cuotas` (`term`) | Term: 36 or 60 monthly instalments. |
| `tipo_interes` (`int_rate`) | Annual interest rate (%). |
| `imp_cuota` (`installment`) | Monthly instalment ($). |
| `rating` / `subrating` (`grade` / `sub_grade`) | Lending Club risk grade: `A` (lowest risk) to `G`; sub-levels `A1` to `G5`. |
| `fico_min` / `fico_max` (`fico_range_low` / `fico_range_high`) | Range of the borrower's FICO score (300–850; higher = better history). |
| `fecha_primer_credito` (`earliest_cr_line`) | Date of their first credit line. |
| `num_lineas_credito` / `num_lineas_credito_total` (`open_acc` / `total_acc`) | Open / total credit lines in the history. |
| `num_hipotecas` (`mort_acc`) | Number of mortgage accounts. |
| `saldo_revolving` (`revol_bal`) | Outstanding revolving credit balance ($). |
| `porc_uso_revolving` (`revol_util`) | % of the revolving limit in use. |
| `porc_uso_tarjetas` (`bc_util`) | % of the limit of their bankcards in use. |
| `porc_tarjetas_75p` (`percent_bc_gt_75`) | % of their bankcards above 75 % of their limit. |
| `num_consultas_6meses` (`inq_last_6mths`) | Credit inquiries in the last 6 months. |
| `num_retrasos_2anios` (`delinq_2yrs`) | Payments more than 30 days late in the last 2 years. |
| `num_meses_desde_ult_retraso` (`mths_since_last_delinq`) | Months since the last late payment. |
| `num_derogatorios` (`pub_rec`) | Derogatory public records (judgments, liens…). |
| `num_quiebras` (`pub_rec_bankruptcies`) | Bankruptcies in public records. |
| `num_cancelaciones_12meses` (`chargeoff_within_12_mths`) | Debts written off in the last 12 months. |
| `estado` (`loan_status`) | Loan outcome: `Fully Paid` (repaid) or `Charged Off` (default). |
| `imp_amortizado` (`total_rec_prncp`) | Principal repaid ($). |
| `imp_recuperado` (`recoveries`) | Amount recovered after default ($). |
| `fecha_concesion` (`issue_d`) | Issue month. |

<a id="s4"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">4 · NULLS</span>

</div>

This section locates the variables with nulls, studies **what the null means** in each one (missing data, data that does not exist, a data-collection problem?) and whether it is **related to default**. Then, variable by variable, it decides how to handle it (decisions D1–D6) and applies the fix.

<a id="s4-1"></a>

### 4.1 · Which variables have nulls and how many?

In [ ]:
# % of nulls per variable (only those with some)
pct_nulos = df.isna().mean().mul(100).sort_values(ascending=False)
pct_nulos = pct_nulos[pct_nulos > 0]
pct_nulos.round(2).to_frame('% nulls')

**What is done:** the % of nulls of each variable is computed and only those with some are shown.

**What we see:** **7 variables** have nulls. `num_meses_desde_ult_retraso` has more than half (51 %); the rest, less than 7 %.

In [ ]:
# Horizontal bar chart with the % of nulls of each variable
fig, ax = plt.subplots(figsize=(8, 3.5))
pct_nulos.sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('% of loans with null')
ax.set_title('Variables with nulls')
for i, valor in enumerate(pct_nulos.sort_values()):
    ax.text(valor + 0.5, i, f'{valor:.1f} %', va='center')   # label with the value
plt.tight_layout()
plt.show()

**What is done:** bar chart with the % of nulls of each variable.

**What we see:** the same as the previous table: one variable with many nulls and six with few.

<a id="s4-2"></a>

### 4.2 · Is the null related to default?

If loans with a null default more (or less) than the rest, the null **carries information** and it is worth keeping it somehow, instead of filling it in as if it were any other value.

In [ ]:
# For each variable with nulls: number of nulls and default rate with and without null
comparativa = pd.DataFrame({
    'n_nulos': df[pct_nulos.index].isna().sum(),
    'tasa_impago_con_nulo': [impago[df[col].isna()].mean() for col in pct_nulos.index],
    'tasa_impago_sin_nulo': [impago[df[col].notna()].mean() for col in pct_nulos.index],
}, index=pct_nulos.index)

# Difference in percentage points (positive = nulls default more)
comparativa['diferencia_pp'] = (comparativa['tasa_impago_con_nulo'] - comparativa['tasa_impago_sin_nulo']) * 100
comparativa

**What is done:** for each variable with nulls, the default rate of loans **with** a null is compared with that of loans **without** one. Columns: `n_nulos` (number of loans with a null); `tasa_impago_con_nulo` and `tasa_impago_sin_nulo` (share of defaults in each group: 0.145 = 14.5 %); `diferencia_pp` (difference between the two rates in **percentage points**: positive = nulls default more; e.g. 20.3 % − 14.5 % = +5.8 pp).

**What we see:** nulls in `empleo` and `antigüedad_empleo` default **about 6 points more**: the null carries a lot of signal. In cards and mortgages the difference is +1.6 to +1.8 points, and in `porc_uso_revolving` +3.3 (but only 221 loans). In `num_meses_desde_ult_retraso` nulls default slightly **less** (−0.7): consistent with the null meaning "no late payments".

In [ ]:
# Chart: default rate with and without null, compared with the overall rate
fig, ax = plt.subplots(figsize=(9, 3.5))
comparativa[['tasa_impago_con_nulo', 'tasa_impago_sin_nulo']].mul(100).plot.bar(
    ax=ax, color=['indianred', 'steelblue'], rot=20)
ax.axhline(TASA_IMPAGO_GLOBAL * 100, color='grey', linestyle='--', label='Overall rate')  # reference
ax.set_ylabel('% default')
ax.set_title('Default rate depending on whether the variable is null')
ax.legend(['Overall rate', 'With null', 'Without null'])
plt.tight_layout()
plt.show()

**What is done:** chart with the default rate with null (red) and without null (blue) for each variable; the dashed line is the overall rate.

**What we see:** the red bars of `empleo` and `antigüedad_empleo` stand clearly above the overall rate.

<a id="s4-3"></a>

### 4.3 · Do nulls depend on the issue year?

Notebook 01 showed that some variables were not collected in the early years. If nulls are concentrated in a specific year, they are **structural** (they do not depend on the borrower).

In [ ]:
# % of nulls per issue year for each variable with nulls
anio = df['fecha_concesion'].dt.year
df[pct_nulos.index].isna().groupby(anio).mean().mul(100).round(2).T

**What is done:** % of nulls of each variable per issue year (rows: variables; columns: years).

**What we see:** the card variables have ~15 % nulls in 2012 and ~1 % afterwards, and `num_hipotecas` only has nulls in 2012: it points to a **data-collection** problem in 2012 (detailed in 4.4.5). `empleo` and `antigüedad_empleo` have nulls every year (~4–7 %), so they are not structural.

<a id="s4-4"></a>

### 4.4 · Variable by variable

Each variable with nulls is studied separately, in the order in which it is fixed: first the observation, then the decision (**Decision Dx** box) and finally the code that applies it.

<a id="s4-4-1"></a>

#### 4.4.1 · `num_meses_desde_ult_retraso` (D1)

**What it is:** months since the borrower's last late payment (*delinquency*), according to their credit report (original: `mths_since_last_delinq`). It is empty when no late payment is recorded.

**Related variable:** `num_retrasos_2anios` (`delinq_2yrs`), number of payments more than 30 days late in the last 2 years; it is used to check what the null means.

**Question:** what does it mean for it to be null?

In [ ]:
# Late payments in the last 2 years depending on whether num_meses_desde_ult_retraso is null
es_nulo = df['num_meses_desde_ult_retraso'].isna()
df.groupby(es_nulo)['num_retrasos_2anios'].agg(
    media='mean',
    pct_con_algun_retraso=lambda s: (s > 0).mean() * 100,
).rename(index={False: 'with value', True: 'null'})

**What is done:** the number of late payments in the last 2 years is compared between loans with a value and with a null in `num_meses_desde_ult_retraso`. `media`: average number of late payments; `pct_con_algun_retraso`: % with at least one.

**What we see:** among the nulls, only **0.5 %** have any late payment in 2 years (versus 39 % of those with a value): the null means **"no recorded late payment"**.

In [ ]:
# Distribution of the non-null values (are there many zeros?)
fig, ax = plt.subplots(figsize=(8, 3))
df['num_meses_desde_ult_retraso'].dropna().plot.hist(bins=60, ax=ax, color='steelblue')
ax.set_xlabel('Months since the last late payment')
ax.set_title('Distribution of num_meses_desde_ult_retraso (without nulls)')
plt.tight_layout()
plt.show()

print(f"Loans with value 0: {(df['num_meses_desde_ult_retraso'] == 0).sum():,}")

**What is done:** histogram of the months since the last late payment (without nulls) and number of loans with value 0.

**What we see:** values are spread between 0 and about 84 months and almost disappear beyond that. Only 451 loans have a late payment in the same month (value 0).

In [ ]:
# Range of values: minimum, maximum and percentiles (nulls are not counted)
df['num_meses_desde_ult_retraso'].describe(percentiles=[.01, .25, .5, .75, .95, .99, .999])

**What is done:** statistics of the variable: mean, minimum, maximum and percentiles (the value below which that % of loans falls).

**What we see:** median of **31 months**; 99 % is below 80 months, but the maximum reaches 188.

In [ ]:
# Right tail: how many loans exceed each limit? (84 months = 7 years)
for limite in [80, 84, 100, 120]:
    print(f"More than {limite} months: {(df['num_meses_desde_ult_retraso'] > limite).sum():,}")

**What is done:** we count the loans above several limits (84 months = 7 years).

**What we see:** only **289** exceed 84 months. This fits with late payments being removed from the credit report after 7 years.

In [ ]:
# Default rate by fine bands of months since the last late payment (including the null as a band)
tramos_finos = pd.cut(df['num_meses_desde_ult_retraso'],
                      bins=[-1, 0, 6, 12, 24, 36, 48, 60, 72, 84, 200],
                      labels=['0', '1-6', '7-12', '13-24', '25-36', '37-48', '49-60', '61-72', '73-84', '>84'])
tramos_finos = tramos_finos.cat.add_categories('null').fillna('null')
pd.DataFrame({
    'n_prestamos': tramos_finos.value_counts(sort=False),
    'tasa_impago': impago.groupby(tramos_finos, observed=False).mean(),
})

**What is done:** default rate (`tasa_impago`) and number of loans (`n_prestamos`) by fine bands of months, including the null as one more band.

**What we see:** the default rate barely changes between bands (from 14 % to 17 %) and the null is at 14.5 %: the variable has **little signal**.

**Consistency with `num_retrasos_2anios`** (late payments in the last 24 months): if the null means "no late payment", nulls should have 0 late payments in 2 years; and whoever was late less than 24 months ago should have at least 1.

In [ ]:
# Nulls versus late payments in the last 2 years
pd.crosstab(
    es_nulo.map({True: 'null', False: 'with value'}),
    df['num_retrasos_2anios'] > 0,
    rownames=['num_meses_desde_ult_retraso'],
    colnames=['late payments in the last 2 years?'],
)

**What is done:** cross table: null or with value versus whether there were late payments in the last 2 years.

**What we see:** of the 226,509 nulls, **225,322 have no late payments** in 2 years and **1,187 do** (0.5 %): those few are inconsistent with "no late payment".

In [ ]:
# Loans with a value: % with late payments in 2 years depending on whether the last one was less than 24 months ago
con_valor = df[~es_nulo]
reciente = con_valor['num_meses_desde_ult_retraso'] < 24
(con_valor['num_retrasos_2anios'] > 0).groupby(reciente).mean().mul(100) \
    .rename(index={True: 'less than 24 months', False: '24 months or more'}) \
    .to_frame('% with late payments in 2 years')

**What is done:** among loans with a value, % with late payments in 2 years depending on whether the last late payment was less than 24 months ago.

**What we see:** if the last late payment was less than 24 months ago, **99.9 %** have late payments in 2 years; if it was longer ago, only 0.7 %. Both variables are consistent with each other.

> **Decision D1 · Convert `num_meses_desde_ult_retraso` into 4 bands: `sin_retraso`, `0-24`, `25-48` and `>48`.**
> - **What the null means:** that no late payment is recorded. More than 99 % of the nulls have 0 late payments in the last 2 years. Nuance: late payments disappear from the credit report after 7 years (84 months, exactly where the distribution ends), so it means "no **recorded** late payment", not "never in their life".
> - **Why bands:** no number of months represents "no late payment". Imputing 0 would say the opposite (a late payment this month) and imputing an extreme value (e.g. 999) would invent data.
> - **Assignment:**
>   - Loans with a value → the band matching their months.
>   - Null and 0 late payments in 2 years → `sin_retraso` (no late payment).
>   - Null **with** late payments in 2 years (inconsistency, about 0.5 % of the nulls) → `0-24`, because `num_retrasos_2anios` proves there was a late payment in the last 24 months.
> - **Cut-offs:** 24 is the window of `num_retrasos_2anios`. 48 splits the rest into two groups of similar size (arbitrary cut-off). The last band is open (`>48`), so it also covers the tail beyond 84 months without having to cap it.
> - **To keep in mind:** default barely changes between bands (between 14 % and 16 %), so it is probably a weakly predictive variable. It will be checked in the EDA.
> - **In production:** the function `tramo_ultimo_retraso` will be applied in the same way to validation and in the API.

In [ ]:
def tramo_ultimo_retraso(meses, retrasos_2anios):
    '''Assigns the band of months since the last late payment (decision D1).

    meses           : Series with num_meses_desde_ult_retraso (NaN = no recorded late payment)
    retrasos_2anios : Series with num_retrasos_2anios (late payments in the last 24 months)
    Returns an ordered categorical variable with the bands.
    '''
    # Conditions in order: np.select assigns the FIRST one that is met in each row
    condiciones = [
        meses.isna() & (retrasos_2anios == 0),   # no recorded late payment
        meses.isna() & (retrasos_2anios > 0),    # inconsistent null: there was a late payment in the last 24 months
        meses <= 24,                             # recent late payment
        meses <= 48,                             # intermediate late payment
        meses > 48,                              # old late payment (includes the >84-month tail)
    ]
    tramos = ['sin_retraso', '0-24', '0-24', '25-48', '>48']
    # Rows that meet no condition (e.g. both values empty) -> empty text -> NaN in the categorical
    resultado = np.select(condiciones, tramos, default='')
    return pd.Categorical(resultado, categories=['sin_retraso', '0-24', '25-48', '>48'], ordered=True)

In [ ]:
# Create the new variable from the original one
df['tramo_ult_retraso'] = tramo_ultimo_retraso(df['num_meses_desde_ult_retraso'], df['num_retrasos_2anios'])

# Check: every loan has a band
assert df['tramo_ult_retraso'].notna().all(), "There are loans without a band"

# Loans and default rate by band
pd.DataFrame({
    'n_prestamos': df['tramo_ult_retraso'].value_counts(sort=False),
    'tasa_impago': impago.groupby(df['tramo_ult_retraso'], observed=False).mean(),
})

**What is done:** `tramo_ult_retraso` is created with the D1 function, we check that every loan has a band and show the default rate by band.

**What we see:** 225,322 loans `sin_retraso` and the rest spread over the other three bands. Default goes from 14.5 % to 15.5 %: it confirms that it is a weak variable.

In [ ]:
# Drop the original variable: its information is already captured in the band
df = df.drop(columns='num_meses_desde_ult_retraso')
print(f"Columns: {df.shape[1]}")

**What is done:** the original variable, already captured in the band, is dropped.

**What we see:** there are still **34 columns** (`tramo_ult_retraso` comes in and `num_meses_desde_ult_retraso` goes out).

<a id="s4-4-2"></a>

#### 4.4.2 · `antigüedad_empleo` (D2)

**What it is:** years the borrower has been in their current job, as stated in the application (original: `emp_length`). It is text with 11 categories: `< 1 year`, `1 year`, `2 years` … `9 years` and `10+ years`.

In [ ]:
# Loans and default rate per category (including the null)
categoria = df['antigüedad_empleo'].fillna('NULL')
pd.DataFrame({
    'n_prestamos': categoria.value_counts(),
    'tasa_impago': impago.groupby(categoria).mean(),
}).sort_values('tasa_impago', ascending=False)

**What is done:** number of loans and default rate of each job-tenure category, with the null as one more category.

**What we see:** the null has the **highest default rate (20.6 %)**; `10+ years`, the lowest (13.7 %); between 1 and 9 years default barely changes (~15 %).

**Does it match the null of `empleo`?** If they almost always go together, the null is not data lost at random, but a borrower who **did not state any job**.

In [ ]:
# Cross of nulls: job tenure versus job title
nulo_antiguedad = df['antigüedad_empleo'].isna()
nulo_empleo = df['empleo'].isna()
pd.crosstab(
    nulo_antiguedad.map({True: 'tenure null', False: 'tenure with value'}),
    nulo_empleo.map({True: 'job null', False: 'job with value'}),
    margins=True, margins_name='Total',
)

**What is done:** cross table of the nulls of `antigüedad_empleo` and `empleo`.

**What we see:** **25,123** loans have both variables null; only 197 have only the tenure null. That is, ~99 % of the tenure nulls are also job nulls.

In [ ]:
# Loans and default rate of each combination of nulls
combinacion = pd.Series(
    np.select(
        [nulo_antiguedad & nulo_empleo, nulo_antiguedad & ~nulo_empleo, ~nulo_antiguedad & nulo_empleo],
        ['both null', 'only tenure null', 'only job null'],
        default='both with value',
    ),
    index=df.index,
)
pd.DataFrame({
    'n_prestamos': combinacion.value_counts(),
    'tasa_impago': impago.groupby(combinacion).mean(),
})

**What is done:** number of loans and default rate of each combination of nulls.

**What we see:** when both values are missing, default is **20.7 %**, versus 14.5 % for those who state both.

**What profile do borrowers with a null tenure have?**

In [ ]:
# Medians of income, DTI, FICO and interest rate depending on whether the tenure is null
df.groupby(nulo_antiguedad).agg(
    ingresos=('ingresos', 'median'),
    dti=('dti', 'median'),
    fico=('fico_min', 'median'),
    tipo_interes=('tipo_interes', 'median'),
).rename(index={False: 'with value', True: 'null'})

**What is done:** medians of income, DTI, FICO and interest rate depending on whether the tenure is null.

**What we see:** borrowers with a null have **much lower income** (42,000 versus 63,000) and slightly higher DTI; FICO is the same.

In [ ]:
# % of loans with verified income and owned outright home depending on whether the tenure is null
pd.DataFrame({
    '% verified income': df['ingresos_verificados'].eq('Verified').groupby(nulo_antiguedad).mean() * 100,
    '% owned outright': df['vivienda'].eq('OWN').groupby(nulo_antiguedad).mean() * 100,
}).rename(index={False: 'with value', True: 'null'})

**What is done:** % of loans with verified income and with a home owned outright, depending on whether the tenure is null.

**What we see:** Lending Club **verifies the income of borrowers with a null much more often** (71 % versus 31 %) and they own their home outright twice as often (19 % versus 9 %).

In [ ]:
# Is it a structural null? % of nulls per issue year
nulo_antiguedad.groupby(df['fecha_concesion'].dt.year).mean().mul(100).round(1).to_frame('% nulls')

**What is done:** % of tenure nulls per issue year, to see whether it is a structural null.

**Observations:**
- Almost all nulls of `antigüedad_empleo` (around 99 %) are also nulls of `empleo`: the borrower **did not state any job**.
- It is the group with the **highest default rate**, around 21 % versus 14.5 % for those who state both values.
- Its profile is different: **lower income** (median of about 42,000 versus 63,000), slightly higher DTI, twice as many **homes owned outright** (around 20 % versus 10 %) and Lending Club **verifies their income much more often** (around 72 % versus 31 %). The FICO score is the same.
- This profile fits **retirees, unemployed people or people whose income does not come from a job** (pensions, rents...). It is a hypothesis consistent with the data, although the data cannot fully confirm it.
- It is not a structural null: it is present every year (between 3.5 % and 7 %).
- Among those who state a tenure, default barely changes from 1 to 9 years, and `10+ years` is the category with the lowest default.

> **Decision D2 · Replace the null of `antigüedad_empleo` with its own category: `sin_empleo_declarado` (no job stated).**
> - **Why:** the null identifies a group with its own profile and the highest default risk. An own category keeps that signal and describes what actually happens.
> - **Discarded alternatives:**
>   - Imputing the mode (`10+ years`): it would assign the **lowest**-risk category to the **highest**-risk group.
>   - Converting it to a number (0–10) with a null flag: it works, but uses two columns for what one category expresses on its own.
> - **Consequence for `empleo`:** since its null almost matches this one, the risk signal of "not stating a job" is captured here (taken into account when deciding on `empleo`).
> - **Pending for the EDA:** consider grouping the 1-to-9-year categories, which behave very similarly.
> - **In production:** in the demo form it will be one more option in the drop-down.

In [ ]:
# Category for borrowers who do not state a job (will be reused in validation and in the API)
CATEGORIA_SIN_EMPLEO = 'sin_empleo_declarado'

# Replace the null with its own category
df['antigüedad_empleo'] = df['antigüedad_empleo'].fillna(CATEGORIA_SIN_EMPLEO)

# Check: no nulls left
assert df['antigüedad_empleo'].notna().all(), "There are nulls left in antigüedad_empleo"

# Loans and default rate per category after the fix
pd.DataFrame({
    'n_prestamos': df['antigüedad_empleo'].value_counts(),
    'tasa_impago': impago.groupby(df['antigüedad_empleo']).mean(),
}).sort_values('tasa_impago', ascending=False)

**What is done:** the null is replaced with the category `sin_empleo_declarado`, we check that no nulls are left and show the default rate per category.

**What we see:** the new category has 25,320 loans and the highest default rate (20.6 %), the same as the null before the fix.

<a id="s4-4-3"></a>

#### 4.4.3 · `empleo` (D3)

**What it is:** job title that the borrower writes in the application (original: `emp_title`), as free text: *"RN"*, *"Teacher"*, *"Software Engineer"*…

Free text written by the borrower. Its null is already captured in `sin_empleo_declarado` (D2). The question is whether the **occupation** carries information about risk and whether it can be grouped into something usable.

In [ ]:
def normalizar_empleo(empleo):
    '''Lowercases, replaces punctuation with spaces and removes extra spaces.
    This way 'RN', 'rn' and ' R.N. ' are treated the same.'''
    return (empleo.str.lower()
                  .str.replace(r'[^a-z0-9]+', ' ', regex=True)   # anything that is not a letter or digit -> space
                  .str.strip())

empleo_norm = normalizar_empleo(df['empleo'])
print(f"Distinct values (original text):   {df['empleo'].nunique():,}")
print(f"Distinct values (normalised text): {empleo_norm.nunique():,}")

**What is done:** a function that normalises the job text (lowercase, no punctuation) is defined and we count the distinct values before and after.

**What we see:** normalising reduces the distinct values from 170,605 to **140,870**: it is still very scattered text.

In [ ]:
# Concentration: % of loans (with a stated job) covered by the most frequent titles
frecuencias = empleo_norm.value_counts()
total_declarado = frecuencias.sum()
for n in [10, 50, 100, 500, 1000]:
    print(f"Top {n:>5} titles: {frecuencias.head(n).sum() / total_declarado:.1%} of the loans")

# Titles that appear only once (the "long tail")
unicos = (frecuencias == 1).sum()
print(f"\nTitles that appear only once: {unicos:,} ({unicos / total_declarado:.1%} of the loans)")

**What is done:** we compute the % of loans (with a stated job) covered by the most frequent titles and how many titles appear only once.

**What we see:** the 100 most frequent titles only cover **24.7 %** of the loans, and **27.6 %** have a title that does not repeat.

In [ ]:
# Default rate of the 30 most frequent titles
top_titulos = frecuencias.head(30).index
tabla_titulos = pd.DataFrame({
    'n_prestamos': frecuencias.head(30),
    'tasa_impago': impago.groupby(empleo_norm).mean().loc[top_titulos],
}).sort_values('tasa_impago')

# Horizontal bar chart with the overall rate as a reference
fig, ax = plt.subplots(figsize=(8, 8))
tabla_titulos['tasa_impago'].mul(100).plot.barh(ax=ax, color='steelblue')
ax.axvline(TASA_IMPAGO_GLOBAL * 100, color='grey', linestyle='--', label='Overall rate')
ax.set_xlabel('% default')
ax.set_title('Default rate of the 30 most frequent job titles')
ax.legend()
plt.tight_layout()
plt.show()

**What is done:** chart with the default rate of the 30 most frequent job titles; the dashed line is the overall rate.

**Observations:**
- There is **real signal**: among the frequent titles, default goes roughly from 7 % (attorneys) to 22 % (servers), while the average is 15 %.
- But the text is very scattered: the 100 most frequent titles only cover about 25 % of the loans, and almost 30 % of the loans have a title that does not repeat. As it is, the variable is unusable.
- **Idea:** group the titles into **sectors** to keep the signal with a manageable number of categories.

##### Grouping by sector · first version of the rules

- **Method:** keyword rules (regular expressions) evaluated **in order**: each title is assigned to the **first** sector whose rule matches. That is why specific rules go before general ones (for example, *nurse manager* must go to health and not to management).
- **Why rules and not more complex NLP techniques:** they are transparent, can be reviewed one by one and are applied the same way in the API.
- Nulls go to `sin_empleo_declarado` (consistent with D2) and whatever matches no rule, to `otros` (other).

In [ ]:
# Rules by sector, in priority order: the FIRST one that matches wins
REGLAS_SECTOR = [
    ('salud',                r'nurse|\brn\b|\blpn\b|\bcna\b|doctor|physician|medical|dental|dentist|pharm|therap|hospital|health|clinic|surgeon|paramedic|\bemt\b|radiolog|caregiver|patient|optometr|veterinar'),
    ('educacion',            r'teacher|professor|school|educat|instructor|tutor|principal|faculty|universit|college|librar'),
    ('legal',                r'attorney|lawyer|paralegal|legal|\bcounsel\b|law clerk|judge'),
    ('seguridad_publica',    r'police|sheriff|deputy|correction|firefight|military|\barmy\b|\bnavy\b|air force|marine|usps|postal|letter carrier|mail carrier|federal|government|state of|county|city of|security guard|security officer|soldier|sergeant|trooper'),
    ('transporte',           r'driver|truck|transport|pilot|logistic|courier|delivery|\bbus\b|dispatch|flight attendant|airline|railroad|conductor'),
    ('hosteleria',           r'^server$|waiter|waitress|bartender|\bcook\b|\bchef\b|restaurant|hotel|barista|food service|dishwasher|housekeep|casino|hospitality'),
    ('finanzas',             r'accountant|accounting|bookkeep|\bbank|financ|teller|\bloan|underwrit|audit|\btax\b|insurance|controller|\bcpa\b|payroll|actuar|credit analyst|accounts payable|accounts receivable|mortgage|investment'),
    ('tecnologia',           r'engineer|developer|software|programmer|\bit\b|network|systems|database|\bdata (?!entry)|\bweb\b|technology|computer|devops|architect'),
    ('comercio',             r'sales|retail|cashier|store|merchand|customer service|account manager|account executive|realtor|real estate|\bbuyer\b|\bcsr\b|walmart|target|costco|home depot|lowe'),
    ('oficios_industria',    r'mechanic|electrician|plumb|carpenter|welder|construction|machinist|maintenance|operator|laborer|warehouse|technician|\btech\b|installer|hvac|painter|foreman|custodian|janitor|landscap|assembl|production|forklift|material handler|lineman|superintendent|inspector'),
    ('autonomo_propietario', r'owner|self employed|selfemployed|proprietor|founder|\bpartner\b|entrepreneur'),
    ('administrativo',       r'admin|assistant|secretary|clerk|receptionist|office|coordinator|data entry|specialist|analyst|representative|associate|processor'),
    ('direccion_gestion',    r'manager|director|supervisor|president|\bceo\b|\bcfo\b|\bcoo\b|\bvp\b|executive|chief|management|team lead|consultant'),
]

def sector_empleo(empleo):
    '''Assigns a sector to each job title according to REGLAS_SECTOR.

    empleo : Series with the original job text (NaN = not stated)
    '''
    texto = normalizar_empleo(empleo).fillna('')
    # One condition per rule; np.select assigns the sector of the FIRST one that is met
    condiciones = [texto.str.contains(patron, regex=True) for _, patron in REGLAS_SECTOR]
    sectores = [sector for sector, _ in REGLAS_SECTOR]
    resultado = np.select(condiciones, sectores, default='otros')
    # Nulls go to the same category as in antigüedad_empleo (D2)
    resultado = np.where(empleo.isna(), CATEGORIA_SIN_EMPLEO, resultado)
    return pd.Series(resultado, index=empleo.index)

# Apply (still as an analysis variable, without modifying df)
sector = sector_empleo(df['empleo'])

In [ ]:
# Loans, % and default rate by sector
tabla_sector = pd.DataFrame({
    'n_prestamos': sector.value_counts(),
    'tasa_impago': impago.groupby(sector).mean(),
})
tabla_sector['% loans'] = tabla_sector['n_prestamos'] / len(df) * 100
cobertura = 1 - (sector == 'otros').sum() / df['empleo'].notna().sum()
print(f"Coverage (loans with a stated job assigned to a sector): {cobertura:.1%}")
tabla_sector.sort_values('tasa_impago')

**What is done:** the titles are grouped into sectors with the rules (first version) and the number of loans, the % and the default rate of each sector are computed. `Coverage`: % of those who state a job that end up assigned to some sector (not to `otros`). Sector names are kept in Spanish (e.g. `tecnologia` = technology, `hosteleria` = hospitality, `otros` = other).

**What we see:** coverage of **73.3 %**. Default goes from **9.6 % (technology)** to **20.3 % (no job stated)**; `otros` gathers 25 % of the loans.

In [ ]:
# Chart: default rate by sector versus the overall rate
fig, ax = plt.subplots(figsize=(8, 5))
tabla_sector['tasa_impago'].sort_values().mul(100).plot.barh(ax=ax, color='steelblue')
ax.axvline(TASA_IMPAGO_GLOBAL * 100, color='grey', linestyle='--', label='Overall rate')
ax.set_xlabel('% default')
ax.set_title('Default rate by job sector')
ax.legend()
plt.tight_layout()
plt.show()

**What is done:** chart of the default rate by sector versus the overall rate.

**What we see:** some sectors are clearly below (technology, legal, education, finance) and others above (transport, hospitality, no job stated).

**Does the sector add anything beyond the grade?** If, within the same grade, default still changes by sector, the sector holds information that the grade does not capture.

In [ ]:
# % default by sector within each grade (A-D hold most of the loans)
impago.groupby([sector, df['rating']]).mean().mul(100).unstack()[['A', 'B', 'C', 'D']] \
      .sort_values('B').round(1)

**What is done:** % default of each sector **within each grade** (A–D, the most frequent).

**What we see:** within the same grade the order is kept. For example, in grade B default goes from 8.1 % (technology) to 16.0 % (hospitality): the sector carries information that the grade does not capture.

##### Review of the rules

To check that the assignment makes sense: the most frequent titles of each sector and those left in `otros`.

In [ ]:
# The 8 most frequent titles of each sector (the column width is increased to read them in full)
pd.set_option('display.max_colwidth', 200)
pd.Series({
    s: ', '.join(f"{titulo} ({n})" for titulo, n in empleo_norm[sector == s].value_counts().head(8).items())
    for s in tabla_sector.sort_values('tasa_impago').index if s not in ('otros', CATEGORIA_SIN_EMPLEO)
}).to_frame('most frequent titles')

**What is done:** the 8 most frequent titles of each sector are shown to check that the assignment makes sense.

**What we see:** assignments are generally correct, but *assistant manager* shows up in `administrativo` (see the review below).

In [ ]:
# The 30 most frequent titles that match no rule
empleo_norm[sector == 'otros'].value_counts().head(30)

**What is done:** the 30 most frequent titles that match no rule (sector `otros`) are shown.

**Result of the review:**
- **Error found:** *assistant manager* falls into `administrativo` because of the word *assistant*, but it is usually a middle manager (for example, in a store) → it should go to `direccion_gestion` (management).
- **Frequent titles in `otros` that clearly fit an existing sector:** *lvn* and *phlebotomist* (health); *usaf*, *department of defense* and *detective* (public safety); *recruiter*, *human resources* and *clerical* (administrative); *claims adjuster* (finance); *labor*, *forman* (typo of *foreman*) and *estimator* (trades).
- **Deliberately left in `otros`:**
  - Social services (*social worker*, *counselor*, *pastor*) and marketing/design (*designer*, *marketing*): no new sectors are created, to keep the list compact.
  - Company names (*AT&T*, *Kaiser Permanente*, *Wells Fargo*, *UPS*...): they say where the borrower works, but not what they do.

##### Grouping by sector · second version of the rules

In [ ]:
# Coverage of the first version (for comparison)
cobertura_v1 = cobertura

# Adjustments after the review: keywords added to each sector
AJUSTES_SECTOR = {
    'salud':             r'|\blvn\b|phlebotom',
    'seguridad_publica': r'|\busaf\b|department of defense|detective',
    'finanzas':          r'|adjuster',
    'oficios_industria': r'|\blabor\b|\bforman\b|estimator',
    'administrativo':    r'|recruit|human resources|\bhr\b|clerical',
}
REGLAS_SECTOR = [(s, patron + AJUSTES_SECTOR.get(s, '')) for s, patron in REGLAS_SECTOR]

# Fix of the error: in 'administrativo', 'assistant' only counts if NOT followed by 'manager'
# (?!.*manager) is a "negative lookahead": the match fails if 'manager' appears later
REGLAS_SECTOR = [(s, patron.replace('assistant', r'assistant(?!.*manager)') if s == 'administrativo' else patron)
                 for s, patron in REGLAS_SECTOR]

# Assign again with the corrected rules
sector = sector_empleo(df['empleo'])
cobertura = 1 - (sector == 'otros').sum() / df['empleo'].notna().sum()
print(f"Coverage: {cobertura_v1:.1%} (first version) -> {cobertura:.1%} (second version)")

# Check of the fixed error
print(f"'assistant manager' now goes to: {sector[empleo_norm == 'assistant manager'].unique()}")

**What is done:** keywords are added to some sectors, the *assistant manager* rule is fixed and the sector is assigned again.

**What we see:** coverage rises from 73.3 % to **74.0 %** and *assistant manager* moves to `direccion_gestion`.

In [ ]:
# Loans, % and default rate by sector with the final rules
tabla_sector = pd.DataFrame({
    'n_prestamos': sector.value_counts(),
    'tasa_impago': impago.groupby(sector).mean(),
})
tabla_sector['% loans'] = tabla_sector['n_prestamos'] / len(df) * 100
tabla_sector.sort_values('tasa_impago')

**What is done:** final table of sectors with the corrected rules.

**What we see:** results barely change compared with the first version: default still goes from 9.6 % (technology) to 20.3 % (no job stated).

> **Decision D3 · Replace `empleo` with `sector_empleo`**: 13 sectors, plus `otros` (not classified) and `sin_empleo_declarado` (null, as in D2), assigned with keyword rules.
> - **Why:** the free text (about 140,000 distinct values) is unusable as it is, but the occupation carries signal: default by sector ranges roughly from 9 % to 20 %, and the differences hold **within the same grade**, so they add information that the grade does not capture.
> - **How:** rules ordered by priority, reviewed with the most frequent titles of each sector and of `otros`. The second version fixes *assistant manager* and adds frequent titles that clearly fitted a sector.
> - **Discarded:** creating sectors for social services or marketing/design, and assigning company names to the company's sector (they stay in `otros`).
> - **Limitation:** about 25 % of those who state a job remain in `otros`, mainly because of the long tail of titles that appear only once.
> - **Ethical note:** occupation is commonly used in credit *scoring*, but it can be correlated with protected characteristics. The model should be monitored so that it does not act as a proxy for them.
> - **In production:** the demo form will ask for the sector in a drop-down. The function `sector_empleo` will be applied in the same way to the validation set.

In [ ]:
# Create the new variable and drop the original free text
df['sector_empleo'] = sector_empleo(df['empleo'])
df = df.drop(columns='empleo')

# Check: no nulls and the expected sectors
assert df['sector_empleo'].notna().all(), "There are loans without a sector"
print(f"Sectors: {df['sector_empleo'].nunique()} · Columns: {df.shape[1]}")

**What is done:** `sector_empleo` is created and the free text `empleo` is dropped.

**What we see:** 15 categories (13 sectors + `otros` + `sin_empleo_declarado`) and still 34 columns.

<a id="s4-4-4"></a>

#### 4.4.4 · `porc_uso_revolving` (D4)

**What it is:** % of the *revolving* credit limit the borrower is using = revolving balance / revolving limit × 100 (original: `revol_util`). **Revolving** credit is credit that can be used, repaid and used again up to a limit: credit cards, store cards and credit lines. A value above 100 means the limit has been exceeded.

**Related variable:** `saldo_revolving` (`revol_bal`), outstanding revolving balance ($): it is the numerator of the ratio.

In [ ]:
# porc_uso_revolving null: what revolving balance do those borrowers have?
df.loc[df['porc_uso_revolving'].isna(), 'saldo_revolving'].describe()

**What is done:** statistics of the revolving balance of the loans with a null `porc_uso_revolving`.

**What we see:** there are 221 loans; at least 75 % have a **zero balance**, but some have very high balances (maximum of $232,186).

In [ ]:
nulo_rev = df['porc_uso_revolving'].isna()
saldo_cero = df['saldo_revolving'].eq(0)

# 1) Nulls versus revolving balance = 0
display(pd.crosstab(nulo_rev, saldo_cero, rownames=['porc_uso_revolving null'], colnames=['saldo_revolving = 0']))

# 2) Consistency: those with a zero balance that DO have a value, what % utilisation do they have?
display(df.loc[saldo_cero & ~nulo_rev, 'porc_uso_revolving'].describe())

# 3) Nulls with a positive balance: how much balance do they have?
df.loc[nulo_rev & ~saldo_cero, ['saldo_revolving', 'num_lineas_credito']].describe().round(0)

**What is done:** (1) cross table of nulls versus zero balance; (2) utilisation of the loans with a zero balance and a value; (3) statistics of the nulls with a positive balance.

**What we see:** 172 nulls have a zero balance and 49 a positive balance. The 1,081 loans with a zero balance **and a value** always have 0 % utilisation. The 49 nulls with a positive balance have high balances (median of $33,907).

In [ ]:
# Four groups depending on whether porc_uso_revolving is null and whether the revolving balance is 0
grupo_rev = pd.Series(
    np.select([nulo_rev & saldo_cero, nulo_rev & ~saldo_cero, ~nulo_rev & saldo_cero],
              ['null · balance 0', 'null · balance > 0', 'with value · balance 0'],
              default='with value · balance > 0'),
    index=df.index, name='grupo')

# Default rate with its 95 % confidence interval (with few cases, the interval is wide)
n = grupo_rev.value_counts()
tasa = impago.groupby(grupo_rev).mean()
margen = 1.96 * np.sqrt(tasa * (1 - tasa) / n)          # standard error of a proportion × 1.96

perfil = pd.DataFrame({
    'n_prestamos': n,
    '% default': tasa * 100,
    'CI95 lower': (tasa - margen) * 100,
    'CI95 upper': (tasa + margen) * 100,
    '% null in porc_uso_tarjetas': df['porc_uso_tarjetas'].isna().groupby(grupo_rev).mean() * 100,
})
# Medians of the credit profile of each group
perfil.join(df.groupby(grupo_rev)[['num_lineas_credito', 'saldo_revolving', 'ingresos', 'fico_min']].median()).round(1)

**What is done:** 4 groups are formed (null/with value × zero/positive balance) and for each one we show the number of loans, the % default with its **95 % confidence interval** (`CI95 lower` and `CI95 upper`: range in which the true rate would lie with 95 % confidence; the fewer the cases, the wider it is) and the credit profile (medians).

**What we see:** both groups of nulls default around **18 %**, but their intervals are very wide (12–24 % and 8–29 %). Both have 100 % nulls in cards, and the zero-balance group has very few credit lines (3).

In [ ]:
# Default rate by decile of porc_uso_revolving: shows the risk zone a null would fall into
# depending on whether it is filled with 0 (first decile) or with the median (deciles 5-6)
deciles = pd.qcut(df['porc_uso_revolving'], q=10)
pd.DataFrame({'n_prestamos': deciles.value_counts(sort=False),
              '% default': impago.groupby(deciles, observed=True).mean() * 100}).round(1)

**What is done:** default rate by **decile** of `porc_uso_revolving`: loans are sorted by utilisation and split into 10 groups of the same size. `(22.3, 33.0]` means "more than 22.3 and up to 33.0".

**What we see:** default rises steadily with utilisation: from **11.2 %** in the lowest decile to **17.2 %** in the highest. A 0 % would place the nulls in the lowest-risk decile; the median (55.2 %), in the middle zone.

> **Decision D4 · Fill the 221 nulls of `porc_uso_revolving` with the training median (55.2 %).**
> - **What the null means:** the variable is a ratio (revolving balance / revolving limit), so it is only missing when **the limit is 0 or unknown**. There are two groups:
>   - **Null with a zero balance (172):** 0 / 0. The borrower probably **has no revolving line at all**: 100 % of them also have a null in cards and their median number of credit lines is 3, versus 10 for the rest.
>   - **Null with a positive balance (49):** they have revolving debt but the limit is missing. Hypothesis: cards without a preset limit (*charge cards*). It fits their profile: median income of about $115,000 and balances of about $34,000.
>   - In contrast, the **1,081 loans with a zero balance and a value** have a limit and do not use it: their 0 % is real data.
> - **Observed risk:** both groups of nulls default around **18 %**, versus 14.8 % overall. With so few cases the confidence interval is wide (12–24 % and 8–29 %), so we cannot claim they default more, but **there is no evidence at all that they are low risk**.
> - **Alternatives considered and discarded:**
>   - **Zero balance → 0 %, positive balance → median:** it treats 0 / 0 as 0 and places the 172 in the lowest-risk decile (11.2 % default), against what is observed.
>   - **100 % (maxed out):** it would match their default rate, but invents an extreme value with no basis.
>   - **Impute + `sin_limite_revolving` flag:** it would keep the difference, but it is a column for 0.05 % of the data (about 40 defaults); the model cannot learn anything reliable from it.
>   - **Removing the 221 loans:** it does not avoid bias, it moves it elsewhere. The nulls are not random (profile with little history and worse grade), so removing them is **selection bias**: the model would never see that profile. Moreover, the rule must be applicable in validation and in the API, where a borrower cannot be "removed": it would have to be rejected without a score or imputed anyway. Removing only makes sense when the data is wrong or the case is out of scope by a business rule, which is not the case.
> - **Why the median:** with 0.05 % of the data no option changes the model noticeably, so consistency matters more. Both groups share the essential point: **utilisation cannot be computed**. The median treats them alike, places them in an intermediate risk zone (deciles 5–6, ~15 %) without inventing a low or an extreme risk, and is a simple rule to apply in validation and in the API. The median is used instead of the mean because the variable has values above 100 % (outliers), which shift the mean.
> - **Order:** it is fixed before the cards (D5), which are imputed from this variable.
> - **In production:** the median is computed on training only and stored (`MEDIANA_USO_REVOLVING`) to apply it in the same way in validation and in the API.

In [ ]:
# Median of porc_uso_revolving computed ONLY on training (will be reused in validation and in the API)
MEDIANA_USO_REVOLVING = df['porc_uso_revolving'].median()

def imputar_uso_revolving(uso_revolving, mediana):
    '''Fills the nulls of porc_uso_revolving with the training median (decision D4).

    uso_revolving : Series with porc_uso_revolving (NaN = revolving limit 0 or unknown)
    mediana       : median of porc_uso_revolving computed on training
    Returns the Series without nulls.
    '''
    return uso_revolving.fillna(mediana)

# Number of nulls before the fix (for the log)
n_nulos_rev = df['porc_uso_revolving'].isna().sum()

# Apply the decision
df['porc_uso_revolving'] = imputar_uso_revolving(df['porc_uso_revolving'], MEDIANA_USO_REVOLVING)

# Check: no nulls left
assert df['porc_uso_revolving'].notna().all(), "There are nulls left in porc_uso_revolving"
print(f"Nulls filled: {n_nulos_rev} · Training median used: {MEDIANA_USO_REVOLVING:.1f} %")

**What is done:** the training median is computed, the imputation function (D4) is defined and applied.

**What we see:** **221 nulls** are filled with **55.2 %** and the variable has no nulls left.

<a id="s4-4-5"></a>

#### 4.4.5 · `porc_uso_tarjetas` and `porc_tarjetas_75p` (D5)

**What they are:** two credit-report variables that only look at **bankcards** (Visa or Mastercard credit cards issued by a bank):
- `porc_uso_tarjetas` (original: `bc_util`): % of the total limit of their bankcards that is spent.
- `porc_tarjetas_75p` (original: `percent_bc_gt_75`): % of their bankcards above 75 % of their limit.

*Example:* cards with limits 5,000, 3,000 and 2,000 and balances 4,500, 600 and 0 → `porc_uso_tarjetas` = 5,100 / 10,000 = **51 %**; only the first one is above 75 % of its limit → `porc_tarjetas_75p` = 1 out of 3 = **33.3 %**. `porc_uso_revolving` is the same idea as `porc_uso_tarjetas`, but with all revolving credit (also store cards and credit lines).

They are imputed from `porc_uso_revolving` (see below), which is why it was fixed first.

In [ ]:
# Nulls in porc_uso_tarjetas
df['porc_uso_tarjetas'].isnull().sum()

**What is done:** the nulls of `porc_uso_tarjetas` are counted.

**What we see:** there are **9,864** nulls.

In [ ]:
# Nulls in porc_tarjetas_75p
df['porc_tarjetas_75p'].isnull().sum()

**What is done:** the nulls of `porc_tarjetas_75p` are counted.

**What we see:** there are **9,848** nulls.

In [ ]:
# Do the nulls of the two card variables appear in the same loans?
nulo_uso = df['porc_uso_tarjetas'].isna()
nulo_75p = df['porc_tarjetas_75p'].isna()
pd.crosstab(nulo_uso, nulo_75p, rownames=['porc_uso_tarjetas null'], colnames=['porc_tarjetas_75p null'])

**What is done:** cross table of the nulls of the two card variables (`True` = null).

**What we see:** **9,586** loans have both null; 278 only `porc_uso_tarjetas` and 262 only `porc_tarjetas_75p`.

In [ ]:
# What are those borrowers like? Compare their revolving balance and utilisation with the rest
df.groupby(nulo_uso)[['saldo_revolving', 'porc_uso_revolving', 'num_lineas_credito']].median() \
  .rename(index={False: 'with value', True: 'null'})

**What is done:** medians of revolving balance, revolving utilisation and credit lines depending on whether `porc_uso_tarjetas` is null.

**What we see:** nulls have a **lower balance** ($8,000 versus $11,100) and **fewer credit lines** (8 versus 11), but a similar revolving utilisation.

**Does the null add information beyond the grade?** Loans with a null default slightly more (16.6 % versus 14.8 %), but they also have a worse grade. If the difference disappears within the same grade, the null adds nothing new. 2012, when the null is structural (~15 % of that year's loans), is separated from the other years (~1 %).

In [ ]:
# Period: 2012 (structural null) versus 2013-2015 (occasional null)
periodo = df['fecha_concesion'].dt.year.map(lambda a: '2012' if a == 2012 else '2013-2015').rename('periodo')

# Number of loans and % default by period and grade, separating with value (False) and null (True)
tabla_rating = (impago.groupby([periodo, df['rating'], nulo_uso.rename('nulo')])
                      .agg(n='size', pct_impago='mean')   # n = loans; pct_impago = default rate
                      .unstack('nulo'))                    # with value / null as columns
tabla_rating['pct_impago'] = tabla_rating['pct_impago'].mul(100).round(1)   # convert to %
tabla_rating

**What is done:** number of loans (`n`) and % default (`pct_impago`) by period and grade, separating loans with a value (`False`) and with a null (`True`).

**What we see:** within the same grade, nulls do not default systematically differently. In 2013–2015 they even default less in grades E–G (with few cases).

In [ ]:
# Spearman correlation (rank-based, robust to outliers) between the credit-utilisation variables
df[['porc_uso_tarjetas', 'porc_tarjetas_75p', 'porc_uso_revolving']].corr(method='spearman').round(2)

**What is done:** **Spearman** correlation between the three credit-utilisation variables. It measures whether two variables rise and fall together (from −1 to 1) using the order of the values, so it is not affected by outliers.

**What we see:** they are strongly related: 0.86 between the two card variables, **0.82** between card utilisation and revolving utilisation, and 0.71 between 75p and revolving.

In [ ]:
# How wrong would each imputation method be? It is simulated on the loans that DO have a value:
# their value is "predicted" and compared with the real one (mean absolute error, in percentage points)
deciles_rev = pd.qcut(df['porc_uso_revolving'], q=10, duplicates='drop')   # 10 bands of equal size
con_valor = df[~nulo_uso & df['porc_uso_revolving'].notna()]               # loans with both values
deciles_cv = deciles_rev[con_valor.index]

for col in ['porc_uso_tarjetas', 'porc_tarjetas_75p']:
    real = con_valor[col].dropna()
    pred_global = real.median()                                              # option 1: global median
    pred_tramo = con_valor[col].groupby(deciles_cv, observed=True) \
                               .transform('median')[real.index]             # option 2: median of its revolving band
    print(f"{col:<20} global median error: {(real - pred_global).abs().mean():5.1f} pp · "
          f"median-by-revolving-band error: {(real - pred_tramo).abs().mean():5.1f} pp")

**What is done:** imputation is simulated on the loans that do have a value: their value is "predicted" with the global median and with the median of their revolving decile, and the **mean absolute error** is measured (how wrong it is on average, in **percentage points**, pp).

**What we see:** the median by decile **halves** the error: from 22.7 to 10.5 pp in `porc_uso_tarjetas` and from 30.4 to 18.8 pp in `porc_tarjetas_75p`.

##### Three null patterns

The two variables are almost always null together, but not always. Each pattern has a different explanation, so it is worth separating them.

In [ ]:
# Null pattern of each loan depending on which card variable is missing
patron_tarjetas = pd.Series(
    np.select([nulo_uso & nulo_75p, nulo_uso & ~nulo_75p, ~nulo_uso & nulo_75p],
              ['both null', 'only porc_uso_tarjetas null', 'only porc_tarjetas_75p null'],
              default='none null'),
    index=df.index, name='patron')

# Number of loans of each pattern by issue year, with its default rate
tabla_patron = pd.crosstab(patron_tarjetas, df['fecha_concesion'].dt.year, margins=True, margins_name='Total')
tabla_patron['% default'] = (impago.groupby(patron_tarjetas).mean() * 100).round(1)
tabla_patron

**What is done:** each loan is assigned its card null pattern, which is counted by year, with the default rate of each pattern.

**What we see:** the "both null" pattern is concentrated in **2012** (5,698 out of 9,586). The two partial patterns are few (262 and 278) and almost all from 2014–2015.

##### Why are there so many nulls in 2012? Detail by month

In the table of nulls by year, 2012 stands out (~15 %). Looking at it by month shows that the cause is not the whole year.

In [ ]:
# % of card and mortgage nulls in each month of 2012
es_2012 = df['fecha_concesion'].dt.year == 2012
mes_2012 = df.loc[es_2012, 'fecha_concesion'].dt.to_period('M')
pd.DataFrame({
    'n_prestamos': mes_2012.value_counts().sort_index(),
    '% null porc_uso_tarjetas': nulo_uso[es_2012].groupby(mes_2012).mean() * 100,
    '% null num_hipotecas': df.loc[es_2012, 'num_hipotecas'].isna().groupby(mes_2012).mean() * 100,
}).round(1)

**What is done:** % of card and mortgage nulls in each month of 2012.

**Observation:** in January and February 2012, 100 % of the data is missing and in March 79 %. From April onwards, only ~1 %, the same as in 2013–2015. Lending Club **started collecting these credit-bureau data around March 2012**. The nulls of `num_hipotecas` are exactly those same loans.

There are, therefore, **two causes of null**:
- **January–March 2012:** the data **exists but was not collected** (a system cause, not the borrower).
- **The rest (~1 % per year):** the data **does not exist**. It is checked below.

In [ ]:
# Profile of the nulls outside January-March 2012 (when the null is not due to data collection)
recogida_completa = df['fecha_concesion'] >= '2012-04-01'
columnas_perfil = ['num_lineas_credito', 'num_lineas_credito_total', 'saldo_revolving', 'porc_uso_revolving',
                   'fico_min', 'ingresos', 'dti']
df[recogida_completa].groupby(nulo_uso[recogida_completa])[columnas_perfil].median() \
    .rename(index={False: 'with value', True: 'null'})

**What is done:** medians of the credit profile of the loans with a value and with a null, excluding January–March 2012.

**Observation:** outside January–March 2012, borrowers with a null have **fewer credit lines** (7 versus 11) and **a lower revolving balance**, but the **same revolving utilisation**. It fits borrowers **without bankcards**: their revolving credit comes from store cards or other lines, and the card ratio is 0 / 0.

##### Logical rules between the two variables

By definition, the two variables cannot take any combination of values:
- If `porc_uso_tarjetas` = 0 (nothing spent), no card exceeds 75 % of its limit → `porc_tarjetas_75p` = 0.
- If `porc_tarjetas_75p` = 100 (all above 75 %), total utilisation is **≥ 75 %**.
- If `porc_tarjetas_75p` = 0 (none above 75 %), total utilisation is **≤ 75 %**.
- If `porc_uso_tarjetas` > 75 %, at least one card exceeds 75 % → `porc_tarjetas_75p` > 0.

They are checked with the complete loans, and we look at the values of the two partial null patterns.

In [ ]:
completos = df[~nulo_uso & ~nulo_75p]

# 1) Utilisation 0 -> % of cards above 75 % (should be 0)
print("porc_tarjetas_75p when porc_uso_tarjetas = 0:")
display(completos.loc[completos['porc_uso_tarjetas'].eq(0), 'porc_tarjetas_75p'].describe().round(2))

# 2) Range of porc_uso_tarjetas depending on whether porc_tarjetas_75p is 0, intermediate or 100
grupo_75p = pd.cut(completos['porc_tarjetas_75p'], bins=[-1, 0, 99.99, 100], labels=['0', 'intermediate', '100'])
display(completos.groupby(grupo_75p, observed=True)['porc_uso_tarjetas'].describe().round(1))

# 3) Utilisation > 75 % with porc_tarjetas_75p = 0 (there should be none)
print(f"Utilisation > 75 % and porc_tarjetas_75p = 0: {(completos['porc_uso_tarjetas'].gt(75) & completos['porc_tarjetas_75p'].eq(0)).sum()}")

# 4) Pattern 'only porc_tarjetas_75p null': what card utilisation do they have?
print("\nporc_uso_tarjetas in the pattern 'only porc_tarjetas_75p null':")
df.loc[~nulo_uso & nulo_75p, 'porc_uso_tarjetas'].describe()

**What is done:** the logical rules are checked with the complete loans: (1) what value 75p has when utilisation is 0; (2) the range of utilisation depending on whether 75p is 0, intermediate or 100; (3) whether there is utilisation > 75 % with 75p = 0; (4) what utilisation the loans with only 75p null have.

**What we see:** with utilisation 0, 75p is almost always 0; with 75p = 0 the maximum utilisation is 75, and with 75p = 100 the minimum is 75; there is no case of utilisation > 75 % with 75p = 0; and the 262 loans with only 75p null all have utilisation = 0.

In [ ]:
# Pattern 'only porc_uso_tarjetas null': how many would get an impossible value if imputed by decile only?
deciles_rev = pd.qcut(df['porc_uso_revolving'], q=10)                         # deciles (revolving without nulls after D4)
mediana_uso_decil = df.groupby(deciles_rev, observed=True)['porc_uso_tarjetas'].transform('median')
solo_uso = nulo_uso & ~nulo_75p
pd.crosstab(
    pd.cut(df.loc[solo_uso, 'porc_tarjetas_75p'], bins=[-1, 0, 99.99, 100], labels=['0', 'intermediate', '100']),
    pd.cut(mediana_uso_decil[solo_uso], bins=[-1, 75, 1000], labels=['≤ 75 %', '> 75 %']),
    rownames=['porc_tarjetas_75p'], colnames=['utilisation imputed by decile'],
)

**What is done:** for the loans with only `porc_uso_tarjetas` null, their known 75p is crossed with the value that imputation by decile alone would give them (≤ 75 % or > 75 %).

**Observation:**
- The rules hold in the data: with utilisation 0, `porc_tarjetas_75p` is 0 in 97.7 % of the cases (the 88 exceptions are probably rounding or credit-bureau inconsistencies); with 75p = 100, the minimum utilisation is 75; with 75p = 0, the maximum is 75; and there is no loan with utilisation > 75 % and 75p = 0.
- **Only `porc_tarjetas_75p` null (262):** all of them have utilisation = 0, so the value is **deduced**: 0.
- **Only `porc_uso_tarjetas` null (278):** imputation by decile alone would give an **impossible** value to part of them (utilisation ≤ 75 % with 75p = 100, or utilisation > 75 % with 75p = 0). It has to be bounded with the known 75p.

##### Risk of the nulls within the same revolving decile

It complements the comparison by grade (previous cells): if imputation places the loan in its decile, does it default like the others in that decile?

In [ ]:
# % default by revolving decile, with value versus null in cards
tabla_dec = (impago.groupby([deciles_rev, nulo_uso.rename('nulo')], observed=True)
                   .agg(n='size', pct_impago='mean')
                   .unstack('nulo'))
tabla_dec['pct_impago'] = tabla_dec['pct_impago'].mul(100).round(1)
tabla_dec

**What is done:** number of loans and % default by revolving decile, separating with value (`False`) and null (`True`) in cards.

**Observation:** within the same revolving decile, nulls default **1–2 points more**, but within the same **grade** they do not (previous grade cell). It is not a contradiction: nulls have a worse grade, and revolving does not capture that difference, while the grade does. Since the model will include the grade, **no null flag is needed**.

##### Extended comparison of imputation methods

The values of 30 % of the complete loans are hidden, predicted with each method (learned on the other 70 %) and the **mean absolute error** (in percentage points) is measured.

**Nuance:** the simulation assumes that the nulls resemble the complete loans. That is true for January–March 2012 (the data exists). For borrowers without cards the "real value" does not exist and any imputation is a filler value, so there the measured accuracy is partly illusory.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor

# Split the complete loans into a part to learn from (70 %) and another to measure the error (30 %)
aprender, medir = train_test_split(completos, test_size=0.3, random_state=42)
variables_gb = ['porc_uso_revolving', 'saldo_revolving', 'num_lineas_credito', 'num_lineas_credito_total',
                'fico_min', 'ingresos', 'dti', 'num_consultas_6meses']

def mediana_por_tramos(col, n_tramos):
    '''Imputes with the median of col in each band of porc_uso_revolving (bands of equal size).'''
    _, limites = pd.qcut(aprender['porc_uso_revolving'], q=n_tramos, retbins=True, duplicates='drop')
    limites[0], limites[-1] = -np.inf, np.inf                                     # open-ended extremes
    medianas = aprender[col].groupby(pd.cut(aprender['porc_uso_revolving'], limites), observed=True).median()
    return pd.cut(medir['porc_uso_revolving'], limites).map(medianas).astype(float)

errores = {}
for col in ['porc_uso_tarjetas', 'porc_tarjetas_75p']:
    real = medir[col]
    error = lambda pred: (real - pred).abs().mean()                              # mean absolute error
    errores[col] = {'global median': error(aprender[col].median())}
    for n in [5, 10, 20, 50]:
        errores[col][f'median by {n} bands'] = error(mediana_por_tramos(col, n))
    regresion = LinearRegression().fit(aprender[['porc_uso_revolving']], aprender[col])
    errores[col]['linear regression (revolving)'] = error(regresion.predict(medir[['porc_uso_revolving']]).clip(0))
    gb = HistGradientBoostingRegressor(loss='absolute_error', max_iter=200, random_state=0)
    errores[col]['gradient boosting (8 variables)'] = error(gb.fit(aprender[variables_gb], aprender[col]).predict(medir[variables_gb]))

pd.DataFrame(errores).round(1)

**What is done:** 7 imputation methods are compared with a simulation: they learn from 70 % of the complete loans and the error is measured on the other 30 %. *Gradient boosting* is a *machine learning* model that combines many decision trees.

**Observation:** deciles halve the error compared with the global median. With more bands the improvement is minimal, linear regression does not beat the deciles and gradient boosting only gains ~1 point at the cost of deploying a second model in the API.

##### How much do these variables add to the prediction?

If they add little, the choice of imputation method weighs little in the final model and simplicity should come first. We measure the **AUC** (0.5 = random, 1 = perfect) of each variable alone and of a logistic regression with the sub-grade, with and without these variables.

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

impago_completos = impago[completos.index].astype(int)

# 1) AUC of each variable on its own (max(AUC, 1 - AUC) is taken so that the direction does not matter)
for col in ['porc_uso_revolving', 'porc_uso_tarjetas', 'porc_tarjetas_75p']:
    auc = roc_auc_score(impago_completos, completos[col])
    print(f"AUC {col:<20}: {max(auc, 1 - auc):.3f}")

# 2) Logistic regression with the sub-grade (A1 = 0 ... G5 = 34) and adding the credit-utilisation variables
subrating_num = completos['subrating'].astype(str).map(lambda s: (ord(s[0]) - ord('A')) * 5 + int(s[1]) - 1)
for extra in [[], ['porc_uso_revolving'], ['porc_uso_revolving', 'porc_uso_tarjetas', 'porc_tarjetas_75p']]:
    X = pd.concat([subrating_num.rename('subrating'), completos[extra]], axis=1)
    auc_cv = cross_val_score(LogisticRegression(max_iter=1000), X, impago_completos, cv=3, scoring='roc_auc').mean()
    print(f"Logistic subrating + {extra}: AUC = {auc_cv:.4f}")

**What is done:** (1) **AUC** of each variable on its own: ability to rank borrowers from more to less risk (0.5 = random, 1 = perfect); (2) AUC of a logistic regression with the sub-grade, with or without the credit-utilisation variables (cross-validation with 3 folds).

**Observation:** each variable on its own has an AUC of ~0.54–0.55 (weak signal) and, with the sub-grade in the model, **they add nothing** (in a linear model). Probably because Lending Club already used these credit-bureau data to compute the grade. Two questions remain for the EDA and modelling: whether it is worth keeping all three variables (almost redundant) and whether the model should use Lending Club's grade (which partly means relying on their own model).

In [ ]:
# Medians of the two variables in each revolving decile: are they possible combinations?
df.groupby(deciles_rev, observed=True)[['porc_uso_tarjetas', 'porc_tarjetas_75p']].median()

**What is done:** median of the two card variables in each revolving decile.

**Observation:** both medians grow together and no pair breaks the logical rules (e.g. 15.5 with 0 in the first decile; 95.4 with 100 in the last one). When both variables are missing, imputation by decile gives consistent combinations.

> **Decision D5 · Impute `porc_uso_tarjetas` and `porc_tarjetas_75p` with the training median of their `porc_uso_revolving` decile, first applying the logical rules between the two variables. No null flag.**
> - **What the null means:** two causes. In **January–March 2012** (~5,700) the data exists but Lending Club did not collect it yet. In the **rest** (~1 % per year) the borrower probably **has no bankcards** (fewer lines and lower balance, same revolving utilisation) and the ratio is 0 / 0.
> - **Why by revolving decile:** the variables are highly correlated with `porc_uso_revolving` (0.82 and 0.71), which is available. The median of the decile halves the error compared with the global median (≈ 10 versus 23 pp and ≈ 19 versus 30 pp). More bands, regression or gradient boosting barely improve and complicate the API. The median (not the mean) is used because of the values above 100 %.
> - **Logical rules (before imputing), by pattern:**
>   - **Both null (9,586):** both with the median of their decile. The medians by decile are consistent combinations.
>   - **Only `porc_tarjetas_75p` null (262):** if utilisation is **0**, 75p is **0** (deduced, not imputed). In training all 262 have utilisation 0. If a case with utilisation > 0 arrived (validation, API), the median of the decile is used and, if utilisation exceeds 75 %, 75p cannot be 0: in that case the median 75p of the loans with utilisation > 75 % is used.
>   - **Only `porc_uso_tarjetas` null (278):** median of the decile **bounded** by the known 75p: if 75p = 100, at least 75 %; if 75p = 0, at most 75 %. Without the bound, part of them would get an impossible value.
> - **No null flag:** within the same grade nulls do not default more; the difference seen within the revolving decile is explained by the grade, which will be in the model.
> - **Alternative considered and discarded (documented): starting the data in April 2012** in notebook 01 (D7) instead of January. It would remove the ~5,700 card collection nulls and all the `num_hipotecas` nulls without imputing anything, at the cost of losing 1.3 % of the data. It would not be selection bias (the criterion is the collection system, not the borrower's profile). It is discarded because it requires redoing notebook 01 and the figures of D1–D4, imputation by decile is reasonable for that 1.3 %, and these variables weigh little in the model. If notebook 01 is redone in the future, it is worth reconsidering.
> - **Pending for the EDA:** the three credit-utilisation variables are almost redundant and, with the sub-grade, add nothing in a linear model (AUC 0.6695 → 0.6699). Consider whether to keep all of them and whether the model should use Lending Club's grade.
> - **In production:** the decile limits (`LIMITES_DECILES_REV`, open-ended), the table of medians by decile (`MEDIANAS_TARJETAS_POR_DECIL`) and `MEDIANA_75P_USO_ALTO` are stored. The function `imputar_tarjetas` is applied in the same way in validation and in the API, **after** D4.

In [ ]:
# --- Values learned ONLY on training (will be reused in validation and in the API) ---------------
# Limits of the porc_uso_revolving deciles (no nulls after D4); open-ended so that
# any new value (e.g. 300 % in the API) falls into the first or the last decile
_, LIMITES_DECILES_REV = pd.qcut(df['porc_uso_revolving'], q=10, retbins=True)
LIMITES_DECILES_REV[0], LIMITES_DECILES_REV[-1] = -np.inf, np.inf

# Median of the two variables in each decile (decile 0 = lowest utilisation ... 9 = highest; nulls are not counted)
decil_entrenamiento = pd.cut(df['porc_uso_revolving'], bins=LIMITES_DECILES_REV, labels=False)
MEDIANAS_TARJETAS_POR_DECIL = df.groupby(decil_entrenamiento)[['porc_uso_tarjetas', 'porc_tarjetas_75p']].median()

# Median of porc_tarjetas_75p when card utilisation exceeds 75 % (for the 75p logical rule)
MEDIANA_75P_USO_ALTO = df.loc[df['porc_uso_tarjetas'] > 75, 'porc_tarjetas_75p'].median()


def imputar_tarjetas(datos, limites, medianas, mediana_75p_uso_alto):
    '''Fills the nulls of porc_uso_tarjetas and porc_tarjetas_75p (decision D5).

    datos                : DataFrame with porc_uso_tarjetas, porc_tarjetas_75p and porc_uso_revolving (no nulls, D4)
    limites              : limits of the porc_uso_revolving deciles (training)
    medianas             : median of the two variables by decile (training)
    mediana_75p_uso_alto : median of porc_tarjetas_75p with card utilisation > 75 % (training)
    Returns a tuple (porc_uso_tarjetas, porc_tarjetas_75p) without nulls.
    '''
    uso_original = datos['porc_uso_tarjetas']
    p75_original = datos['porc_tarjetas_75p']
    uso, p75 = uso_original.copy(), p75_original.copy()

    # Median that corresponds to each loan according to its revolving decile
    decil = pd.cut(datos['porc_uso_revolving'], bins=limites, labels=False)
    mediana_uso = decil.map(medianas['porc_uso_tarjetas'])
    mediana_75p = decil.map(medianas['porc_tarjetas_75p'])

    # --- porc_tarjetas_75p (the rules use the ORIGINAL utilisation, not the imputed one) ---
    nulo_p75 = p75_original.isna()
    p75[nulo_p75 & uso_original.eq(0)] = 0                                 # utilisation 0 -> no card > 75 % (deduced)
    resto = p75.isna()
    p75[resto] = mediana_75p[resto]                                        # rest -> median of the decile
    incoherente = nulo_p75 & uso_original.gt(75) & p75.eq(0)               # utilisation > 75 % requires at least one card > 75 %
    p75[incoherente] = mediana_75p_uso_alto

    # --- porc_uso_tarjetas (the bounds use the ORIGINAL 75p, only when it was known) ---
    nulo_u = uso_original.isna()
    uso[nulo_u] = mediana_uso[nulo_u]                                      # median of the decile
    cota_min = nulo_u & p75_original.eq(100)                               # all cards > 75 % -> utilisation >= 75
    uso[cota_min] = uso[cota_min].clip(lower=75)
    cota_max = nulo_u & p75_original.eq(0)                                 # no card > 75 % -> utilisation <= 75
    uso[cota_max] = uso[cota_max].clip(upper=75)

    return uso, p75


# Apply the decision
df['porc_uso_tarjetas'], df['porc_tarjetas_75p'] = imputar_tarjetas(
    df, LIMITES_DECILES_REV, MEDIANAS_TARJETAS_POR_DECIL, MEDIANA_75P_USO_ALTO)

# Checks: no nulls and no impossible combinations
assert df[['porc_uso_tarjetas', 'porc_tarjetas_75p']].notna().all().all(), "There are nulls left in the card variables"
assert not (df['porc_uso_tarjetas'].eq(0) & df['porc_tarjetas_75p'].gt(0) & nulo_75p).any(), "Utilisation 0 with imputed 75p > 0"
assert not (df['porc_tarjetas_75p'].eq(100) & df['porc_uso_tarjetas'].lt(75)).any(), "75p = 100 with utilisation < 75 %"
assert not (df['porc_tarjetas_75p'].eq(0) & df['porc_uso_tarjetas'].gt(75)).any(), "75p = 0 with utilisation > 75 %"

print(f"Nulls filled · porc_uso_tarjetas: {nulo_uso.sum():,} · porc_tarjetas_75p: {nulo_75p.sum():,}")
print(f"Median 75p with utilisation > 75 % (fallback rule): {MEDIANA_75P_USO_ALTO:.1f}")
MEDIANAS_TARJETAS_POR_DECIL.round(1)

**What is done:** the decile limits, the medians by decile and the fallback median are computed on training; the D5 function with the logical rules is defined; it is applied and we check that no nulls or impossible combinations are left.

**What we see:** **9,864** nulls of `porc_uso_tarjetas` and **9,848** of `porc_tarjetas_75p` are filled, and the 4 checks pass. The final table is the one that will be stored for the API: decile (0 = lowest revolving utilisation, 9 = highest) and the two medians assigned.

<a id="s4-4-6"></a>

#### 4.4.6 · `num_hipotecas` (D6)

**What it is:** number of mortgage accounts of the borrower according to their credit report (original: `mort_acc`).

**Related variable:** `vivienda` (`home_ownership`), housing situation **stated** by the borrower: `RENT`, `MORTGAGE` (owned with a mortgage), `OWN` (owned outright) and other rare values (`NONE`, `OTHER`, `ANY`).

In [ ]:
# Distribution of num_hipotecas (including the null)
df['num_hipotecas'].value_counts(dropna=False).sort_index().head(12)

**What is done:** number of loans with each number of mortgages, including the null (`NaN`).

**What we see:** most have **0 mortgages** (186,022); the number of loans drops as the number of mortgages rises, and there are 5,328 nulls.

In [ ]:
nulo_hip = df['num_hipotecas'].isna()

# 1) Does it match the card null?
display(pd.crosstab(nulo_hip, nulo_uso, rownames=['num_hipotecas null'], colnames=['porc_uso_tarjetas null']))

# 2) Median number of mortgages by housing type (possible imputation value by group)
display(df.groupby('vivienda')['num_hipotecas'].agg(['median', 'mean']).round(2))

# 3) Does the variable carry signal? Default rate by number of mortgages (5 or more grouped)
hip_5 = df['num_hipotecas'].clip(upper=5)
display(pd.DataFrame({'n_prestamos': hip_5.value_counts().sort_index(),
                      'tasa_impago': impago.groupby(hip_5).mean()}))

# 4) Only 2012: within each housing type, do the nulls default differently?
es_2012 = df['fecha_concesion'].dt.year == 2012
t = (impago[es_2012].groupby([df.loc[es_2012, 'vivienda'], nulo_hip[es_2012].rename('nulo')])
                    .agg(n='size', pct_impago='mean').unstack('nulo'))
t['pct_impago'] = t['pct_impago'].mul(100).round(1)
t

**What is done:** (1) match between the mortgage nulls and the card nulls; (2) median and mean number of mortgages by housing type; (3) default by number of mortgages; (4) in 2012, default with and without null within each housing type.

**Observation:**
- The 5,328 nulls **also** have a null in cards: they are loans from January–March 2012 (5,286) and a few from April 2012 (42), when Lending Club did not collect these credit-bureau data yet or was finishing rolling it out (see the cards section).
- The median changes a lot by housing type: **MORTGAGE 2, OWN and RENT 0**, versus a global median of 1.
- The variable has a **clear and ordered signal**: default falls from 17.4 % (0 mortgages) to 11.2 % (5 or more).
- Within each housing type, the 2012 nulls default similarly to those with a value.

##### Does the null depend on the borrower or only on the date?

If, within January–March 2012, loans with and without a value have the same profile, the null is due to the **collection system** and says nothing about the borrower. It is also compared with the rest of 2012 and with 2013–2015.

In [ ]:
# Period of each loan: January-March 2012 (incomplete collection), rest of 2012 and 2013-2015
fecha = df['fecha_concesion']
periodo_recogida = pd.Series(
    np.select([fecha < '2012-04-01', fecha < '2013-01-01'], ['Jan-Mar 2012', 'Apr-Dec 2012'], default='2013-2015'),
    index=df.index, name='periodo')
ene_mar_2012 = periodo_recogida.eq('Jan-Mar 2012')

# 1) Within January-March 2012: profile and default with value versus null
columnas_perfil_hip = ['ingresos', 'dti', 'fico_min', 'num_lineas_credito_total', 'principal']
perfil_hip = df[ene_mar_2012].groupby(nulo_hip[ene_mar_2012])[columnas_perfil_hip].median()
perfil_hip['n_prestamos'] = nulo_hip[ene_mar_2012].value_counts()
perfil_hip['% default'] = impago[ene_mar_2012].groupby(nulo_hip[ene_mar_2012]).mean() * 100
display(perfil_hip.rename(index={False: 'with value', True: 'null'}).round(1))

# 2) Margin of error (95 % CI) of the default rate of the 447 with a value: is the difference significant?
p, n = impago[ene_mar_2012 & ~nulo_hip].mean(), (ene_mar_2012 & ~nulo_hip).sum()
print(f"Default with a value in Jan-Mar 2012: {p:.1%} ± {1.96 * np.sqrt(p * (1 - p) / n):.1%} (n = {n})")

# 3) Are the January-March 2012 borrowers different from those of the other periods?
display(df.groupby(periodo_recogida)[columnas_perfil_hip].median())
display(pd.crosstab(periodo_recogida, df['vivienda'], normalize='index').mul(100).round(1))
pd.crosstab(periodo_recogida, df['rating'], normalize='index').mul(100).round(1)

**What is done:** (1) profile and default of the loans with and without a value **within January–March 2012**; (2) margin of error of the default rate of the 447 with a value; (3) profile, housing and grade of each period.

**Observation:**
- Within January–March 2012, loans **with a value (447) and nulls (5,286) have the same profile** (income, DTI, FICO, credit lines and amount almost identical). The difference in default (14.5 % versus 16.5 %) **is not significant**: with 447 cases the margin of error is about ±3 points.
- Conclusion: the null is **"due to the system"**: it depends on the date and not on the borrower.
- Nuance: the January–March 2012 borrowers are somewhat different from those of other periods (more grade A, higher FICO, more renting). It is a change in the population Lending Club accepted, not an effect of the null.

##### Housing and number of mortgages: do they measure the same thing?

`vivienda` is what the borrower **states**; `num_hipotecas` is what the **credit bureau** says (number of mortgage accounts). If they were the same, one of them would be redundant.

In [ ]:
# Distribution of the number of mortgages (5 or more grouped) within each housing type (% by row)
con_dato_hip = df[~nulo_hip]
display(pd.crosstab(con_dato_hip['vivienda'], con_dato_hip['num_hipotecas'].clip(upper=5),
                    normalize='index').mul(100).round(1))

# Summary by housing type: number of loans, median, mean and "striking" cases
pd.DataFrame({
    'n_prestamos': con_dato_hip['vivienda'].value_counts(),
    'median': con_dato_hip.groupby('vivienda')['num_hipotecas'].median(),
    'mean': con_dato_hip.groupby('vivienda')['num_hipotecas'].mean().round(2),
    '% with 0 mortgages': con_dato_hip['num_hipotecas'].eq(0).groupby(con_dato_hip['vivienda']).mean().mul(100).round(1),
    '% with 1 or more': con_dato_hip['num_hipotecas'].gt(0).groupby(con_dato_hip['vivienda']).mean().mul(100).round(1),
})

**What is done:** distribution of the number of mortgages within each housing type (% by row) and summary by housing type.

**Observation:**
- The medians are **MORTGAGE 2, OWN 0, RENT 0** (NONE, OTHER and ANY: 1, with very few cases).
- Some cases look inconsistent but have an explanation: **9 % of MORTGAGE have 0 mortgages** (e.g. the mortgage is in the partner's name) and **22 % of RENT have some** (e.g. an owned property that is not where they live: investment or second home).
- Conclusion: **they measure different things** and it is worth keeping both variables.

In [ ]:
# % default by number of mortgages (4 or more grouped) within each main housing type
viviendas_principales = con_dato_hip['vivienda'].isin(['MORTGAGE', 'OWN', 'RENT'])
impago[con_dato_hip.index][viviendas_principales] \
    .groupby([con_dato_hip.loc[viviendas_principales, 'vivienda'],
              con_dato_hip.loc[viviendas_principales, 'num_hipotecas'].clip(upper=4)]) \
    .mean().mul(100).round(1).unstack()

**What is done:** % default by number of mortgages within each main housing type.

**Observation:** within **each** housing type, more mortgages = less default (MORTGAGE: from 16.0 % with 0 to 10.7 % with 4 or more; OWN: from 17.4 % to 11.9 %; RENT, flatter: from 17.6 % to 15.5 %). Having mortgages shows that a bank has already granted long-term credit and it is being repaid. The variable carries information that housing alone does not capture.

##### Comparison of imputation methods

As in D5: the values of 30 % of the loans with data are hidden, predicted with each method (learned on the other 70 %) and the **mean absolute error** (in number of mortgages) and the **% of exact hits** are measured. First we look at which variables are most related to the number of mortgages.

In [ ]:
# Years of credit history (from the first credit line to the issue date)
anios_historial = (df['fecha_concesion'] - df['fecha_primer_credito']).dt.days / 365.25

# Spearman correlation of each variable with num_hipotecas (loans with data)
pd.concat([df[['num_hipotecas', 'num_lineas_credito_total', 'ingresos', 'num_lineas_credito', 'fico_min']],
           anios_historial.rename('anios_historial')], axis=1)[~nulo_hip] \
    .corr(method='spearman')['num_hipotecas'].drop('num_hipotecas').sort_values(ascending=False).round(2)

**What is done:** the years of credit history are computed, together with the Spearman correlation of several variables with the number of mortgages.

**What we see:** the most related are total credit lines (0.41), income (0.38) and years of history (0.37).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor

# Loans with data, with the years of history added; 70 % to learn and 30 % to measure
datos_hip = df[~nulo_hip].assign(anios_historial=anios_historial[~nulo_hip])
aprender_h, medir_h = train_test_split(datos_hip, test_size=0.3, random_state=42)
real_h = medir_h['num_hipotecas']

def evaluar(prediccion):
    '''Mean absolute error and % of exact hits (rounding the prediction to an integer).'''
    prediccion = pd.Series(np.asarray(prediccion, dtype=float), index=medir_h.index)
    return {'error_absoluto_medio': (real_h - prediccion).abs().mean(),
            '% exact hits': (real_h == prediccion.round()).mean() * 100}

def tramos(serie_aprender, n):
    '''Limits of n bands of equal size (open-ended extremes).'''
    _, limites = pd.qcut(serie_aprender, q=n, retbins=True, duplicates='drop')
    limites[0], limites[-1] = -np.inf, np.inf
    return limites

def mediana_por_grupos(columnas_grupo):
    '''Median of num_hipotecas by groups; if a group does not exist in "aprender", the median by housing is used.'''
    medianas = aprender_h.groupby(columnas_grupo(aprender_h), observed=True)['num_hipotecas'].median()
    claves = pd.MultiIndex.from_arrays(columnas_grupo(medir_h)) if len(columnas_grupo(medir_h)) > 1 \
             else columnas_grupo(medir_h)[0]
    prediccion = pd.Series(medianas.reindex(claves).to_numpy(), index=medir_h.index)
    return prediccion.fillna(medir_h['vivienda'].map(aprender_h.groupby('vivienda')['num_hipotecas'].median()))

lim_lineas = tramos(aprender_h['num_lineas_credito_total'], 5)       # quintiles of total credit lines
lim_ingresos = tramos(aprender_h['ingresos'], 5)                     # income quintiles

resultados = {
    'global median': evaluar(np.full(len(medir_h), aprender_h['num_hipotecas'].median())),
    'global mean': evaluar(np.full(len(medir_h), aprender_h['num_hipotecas'].mean())),
    'median by housing': evaluar(mediana_por_grupos(lambda d: [d['vivienda']])),
    'housing × total-lines quintile': evaluar(mediana_por_grupos(
        lambda d: [d['vivienda'], pd.cut(d['num_lineas_credito_total'], lim_lineas)])),
    'housing × total lines × income': evaluar(mediana_por_grupos(
        lambda d: [d['vivienda'], pd.cut(d['num_lineas_credito_total'], lim_lineas), pd.cut(d['ingresos'], lim_ingresos)])),
}

# Gradient boosting with 7 numeric variables + housing (encoded as 0/1 columns)
variables_gb_h = ['num_lineas_credito_total', 'num_lineas_credito', 'ingresos', 'anios_historial',
                  'fico_min', 'dti', 'saldo_revolving']
X_aprender = pd.get_dummies(aprender_h[variables_gb_h + ['vivienda']], columns=['vivienda'])
X_medir = pd.get_dummies(medir_h[variables_gb_h + ['vivienda']], columns=['vivienda']).reindex(columns=X_aprender.columns, fill_value=0)
gb_h = HistGradientBoostingRegressor(loss='absolute_error', max_iter=200, random_state=0).fit(X_aprender, aprender_h['num_hipotecas'])
resultados['gradient boosting (8 variables)'] = evaluar(gb_h.predict(X_medir))

pd.DataFrame(resultados).T.round(2)

**What is done:** 6 imputation methods are compared with a simulation (learn on 70 % and measure on 30 %): mean absolute error (`error_absoluto_medio`) in number of mortgages and % of exact hits.

**Observation:**
- The variables most related to the number of mortgages are **total credit lines** (0.41), **income** (0.38) and **years of history** (0.37).
- The **median by housing** lowers the error from ~1.5 to ~1.1 mortgages and **triples the exact hits** (from ~16 % to ~48 %).
- Adding total lines, income or gradient boosting only lowers the error by a further 7–13 %, without improving the hits, at the cost of much larger tables or a second model in the API.
- The global mean (1.67) is worse than the median and is not an integer either.

##### How much does `num_hipotecas` add to the prediction?

If it adds something, it is worth imputing it carefully. A logistic regression with the sub-grade is compared, adding housing, mortgages or both.

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

impago_hip = impago[con_dato_hip.index].astype(int)

# 1) AUC of num_hipotecas on its own (max(AUC, 1 - AUC) so that the direction does not matter)
auc_hip = roc_auc_score(impago_hip, con_dato_hip['num_hipotecas'])
print(f"AUC num_hipotecas alone: {max(auc_hip, 1 - auc_hip):.3f}")

# 2) Logistic: sub-grade (A1 = 0 ... G5 = 34), housing (0/1 columns) and mortgages (10 or more grouped)
subrating_h = con_dato_hip['subrating'].astype(str).map(lambda s: (ord(s[0]) - ord('A')) * 5 + int(s[1]) - 1).rename('subrating')
vivienda_h = pd.get_dummies(con_dato_hip['vivienda'], drop_first=True).astype(int)
hipotecas_h = con_dato_hip['num_hipotecas'].clip(upper=10)
for nombre, X in [('subrating', subrating_h.to_frame()),
                  ('subrating + housing', pd.concat([subrating_h, vivienda_h], axis=1)),
                  ('subrating + mortgages', pd.concat([subrating_h, hipotecas_h], axis=1)),
                  ('subrating + housing + mortgages', pd.concat([subrating_h, vivienda_h, hipotecas_h], axis=1))]:
    auc_cv = cross_val_score(LogisticRegression(max_iter=2000), X, impago_hip, cv=3, scoring='roc_auc').mean()
    print(f"Logistic {nombre:<34}: AUC = {auc_cv:.4f}")

**What is done:** AUC of `num_hipotecas` alone and of a logistic regression with sub-grade, housing and mortgages.

**Observation:** on its own it has an AUC of ~0.55 (weak signal, like almost every variable separately), but, unlike the card variables, **it does add something with the sub-grade**: 0.6691 → 0.6715, and 0.6727 together with housing. The improvement is small but real, so it is worth imputing it with a reasonable method.

**Other details:**
- The medians by housing are **integers**, so the variable remains a count.
- The **NONE, OTHER and ANY** categories have very few cases and will probably be grouped in the categories block. Among the nulls there is only 1 `OTHER`. A **fallback value** is needed for any category without its own median.
- Imputing with medians **slightly lowers the mean** of the variable among the imputed loans; with 1.2 % of the data the effect is minimal.
- There are values of up to **47 mortgages**: they will be reviewed in the outliers block.

> **Decision D6 · Impute `num_hipotecas` with the training median by `vivienda` (MORTGAGE 2, OWN 0, RENT 0...), with the global median (1) as a fallback value. No null flag.**
> - **What the null means:** the 5,328 nulls are from **January–March 2012** (5,286) and April 2012 (42), when Lending Club did not collect this credit-bureau data yet or was finishing rolling it out. Within that period, loans with and without a value have the same profile and a default rate that does not differ significantly: the null depends on the **date**, not on the borrower.
> - **Why by housing:** it is the variable that changes the expected number of mortgages the most (median 2 in MORTGAGE versus 0 in OWN and RENT). It lowers the error from ~1.5 to ~1.1 mortgages and triples the exact hits (16 % → 48 %) compared with the global median. It gives integers and in the API it is a table of a few rows.
> - **Alternatives considered and discarded:**
>   - **Global median (1):** it would give a mortgage to someone who rents and take one away from someone with a mortgage; it hits the exact number three times less often (16 % versus 48 %).
>   - **Global mean (1.67):** worse error and not an integer.
>   - **Housing × total credit lines (× income):** only 7–10 % less error, with no more exact hits, using tables of 30–150 rows.
>   - **Gradient boosting (8 variables):** ~13 % less error, at the cost of deploying a second model in the API.
>   - **Removing January–March 2012:** already considered and discarded in D5.
> - **No null flag:** the null only indicates that the loan is from early 2012. A flag would teach the model a **date**, information that does not exist for a new borrower in the API.
> - **Fallback value:** the training global median (1) for any housing category without its own median (new categories or ones grouped in the categories block), so that the rule never fails.
> - **To keep in mind:** housing and mortgages measure different things (what the borrower states versus what the credit bureau says) and both carry signal: both are kept. Extreme values (up to 47) will be reviewed in the outliers block.
> - **In production:** `MEDIANAS_HIPOTECAS_POR_VIVIENDA` and `MEDIANA_HIPOTECAS_GLOBAL` (computed on training only) are stored and the function `imputar_hipotecas` is applied in the same way in validation and in the API.

In [ ]:
# --- Values learned ONLY on training (will be reused in validation and in the API) ---------------
MEDIANAS_HIPOTECAS_POR_VIVIENDA = df.groupby('vivienda')['num_hipotecas'].median()   # median by housing type (without nulls)
MEDIANA_HIPOTECAS_GLOBAL = df['num_hipotecas'].median()                               # fallback value


def imputar_hipotecas(datos, medianas_vivienda, mediana_global):
    '''Fills the nulls of num_hipotecas (decision D6).

    datos             : DataFrame with num_hipotecas and vivienda
    medianas_vivienda : median of num_hipotecas by housing type (training)
    mediana_global    : global median of num_hipotecas (training), for housing types without their own median
    Returns the num_hipotecas Series without nulls.
    '''
    valor_imputado = datos['vivienda'].map(medianas_vivienda).fillna(mediana_global)   # median of its housing type or fallback
    return datos['num_hipotecas'].fillna(valor_imputado)


# Apply the decision
df['num_hipotecas'] = imputar_hipotecas(df, MEDIANAS_HIPOTECAS_POR_VIVIENDA, MEDIANA_HIPOTECAS_GLOBAL)

# Checks: no nulls and integer values
assert df['num_hipotecas'].notna().all(), "There are nulls left in num_hipotecas"
assert (df['num_hipotecas'] % 1 == 0).all(), "num_hipotecas has non-integer values"

# Value assigned to the nulls according to their housing type
print(f"Nulls filled: {nulo_hip.sum():,} · Fallback value (global median): {MEDIANA_HIPOTECAS_GLOBAL:.0f}")
pd.DataFrame({'mediana_entrenamiento': MEDIANAS_HIPOTECAS_POR_VIVIENDA,
              'nulos_rellenados': df.loc[nulo_hip, 'vivienda'].value_counts()}).fillna(0).astype(int)

**What is done:** the medians by housing type and the fallback global median are computed on training, the D6 function is defined and applied, and we check that no nulls are left and that every value is an integer.

**What we see:** **5,328** nulls are filled (`nulos_rellenados`): 2,221 `MORTGAGE` with 2 mortgages, 2,669 `RENT` and 437 `OWN` with 0, and 1 `OTHER` with 1 (`mediana_entrenamiento` = training median).

<a id="s4-5"></a>

### 4.5 · Final null check

After D1–D6 there should be no nulls left in the training set.

In [ ]:
# Number of nulls per column (must be 0 in every column)
nulos_restantes = df.isna().sum()
assert nulos_restantes.sum() == 0, f"There are nulls left:\n{nulos_restantes[nulos_restantes > 0]}"
print(f"No nulls · Loans: {df.shape[0]:,} · Columns: {df.shape[1]}")

**What is done:** we check that no null is left anywhere in the dataset.

**What we see:** **0 nulls**: 443,469 loans and 34 columns.

<a id="s4-6"></a>

### 4.6 · Summary of the nulls block

| # | Variable | Nulls (training) | What the null means | Treatment | Stored for validation and the API |
|---|---|---|---|---|---|
| D1 | `num_meses_desde_ult_retraso` | 226,509 (51.1 %) | No recorded late payment | Replaced with `tramo_ult_retraso`: `sin_retraso`, `0-24`, `25-48`, `>48` | `tramo_ultimo_retraso` |
| D2 | `antigüedad_empleo` | 25,320 (5.7 %) | The borrower did not state a job (own profile, 20.6 % default) | Category `sin_empleo_declarado` | `CATEGORIA_SIN_EMPLEO` |
| D3 | `empleo` | 28,744 (6.5 %) | Same as D2 | Replaced with `sector_empleo`: 13 sectors + `otros` + `sin_empleo_declarado` | `REGLAS_SECTOR`, `normalizar_empleo`, `sector_empleo` |
| D4 | `porc_uso_revolving` | 221 (0.05 %) | Revolving limit 0 or unknown: utilisation cannot be computed | Training median (55.2 %) | `MEDIANA_USO_REVOLVING`, `imputar_uso_revolving` |
| D5 | `porc_uso_tarjetas`, `porc_tarjetas_75p` | 9,864 and 9,848 (2.2 %) | January–March 2012: not collected; rest: borrower without bankcards | Logical rules + median of their `porc_uso_revolving` decile | `LIMITES_DECILES_REV`, `MEDIANAS_TARJETAS_POR_DECIL`, `MEDIANA_75P_USO_ALTO`, `imputar_tarjetas` |
| D6 | `num_hipotecas` | 5,328 (1.2 %) | January–April 2012: not collected (depends on the date, not the borrower) | Median by `vivienda` (fallback: global median = 1) | `MEDIANAS_HIPOTECAS_POR_VIVIENDA`, `MEDIANA_HIPOTECAS_GLOBAL`, `imputar_hipotecas` |

**Common criteria:**
- **First understand what the null means**, then decide: data missing because of a collection failure (D5 and D6), data that does not exist (D4 and part of D5) and a null with its own meaning (D1, D2 and D3) are not the same thing.
- **No loan is ever removed:** the API has to be able to score any borrower, so every rule must also be applicable to a new borrower.
- **No null flags:** when the null carries signal, it becomes a category (D1, D2, D3); when it does not, or the grade explains it, it is imputed without a flag (D4, D5, D6).
- **Everything is learned on training only** (medians, deciles, rules) and applied in the same way in validation and in the API.

**Order of application** (in validation and in the API): D1, D2 and D3 are independent → D4 → D5 (needs `porc_uso_revolving` without nulls) → D6 (needs `vivienda`; if housing categories are grouped in the categories block, those without their own median get the fallback value).

**Result:** 443,469 loans, 34 columns and **0 nulls**. Two columns change: `num_meses_desde_ult_retraso` → `tramo_ult_retraso` and `empleo` → `sector_empleo`.

**Pending for the EDA:**
- `tramo_ult_retraso` looks weakly predictive (D1).
- Consider grouping `antigüedad_empleo` from 1 to 9 years (D2).
- Check that `sector_empleo` does not act as a proxy for protected characteristics (D3).
- `porc_uso_revolving`, `porc_uso_tarjetas` and `porc_tarjetas_75p` are almost redundant and, with the sub-grade, add nothing in a linear model; also decide whether the model will use Lending Club's grade (D5).

**Pending for the outliers block:** values of `porc_uso_revolving` above 100 % (up to 892 %) and of `num_hipotecas` up to 47.